In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict

# Load the data
file_path = r"C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv"
df = pd.read_csv(file_path)

# Combine Date and Time into a datetime column
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df.set_index('Datetime', inplace=True)

# Filter data from 2013 to 2025
start_date = pd.to_datetime('2013-01-01')
end_date = pd.to_datetime('2025-12-31')
df = df.loc[start_date:end_date]

# Adjust the index to group by NY trading day (18:00 to 17:00)
df['NY_Trading_Day'] = (df.index - pd.Timedelta(hours=7)).normalize()
dates = df['NY_Trading_Day'].unique()

In [2]:
## Hourly Level Retracement Analysis - NY Futures Session (8am-4pm EST)
## Tracks: hourly opens & midpoints from 6pm-7am, distance buckets, timing, London midpoint context


# Define hourly levels to track (6pm previous day through 7am current day)
# Format: (hour, level_type) where hour is 0-23, level_type is 'open' or 'midpoint'
HOURLY_LEVELS = []

# Previous day: 18:00 (6pm) through 23:00 (11pm)
for hour in range(18, 24):
    HOURLY_LEVELS.append((hour, 'open', 'prev_day'))
    HOURLY_LEVELS.append((hour, 'midpoint', 'prev_day'))

# Current day: 00:00 (midnight) through 07:00 (7am)
for hour in range(0, 8):
    HOURLY_LEVELS.append((hour, 'open', 'current_day'))
    if hour < 7:  # 7-8am midpoint is included, but not 8am open
        HOURLY_LEVELS.append((hour, 'midpoint', 'current_day'))

# Distance buckets (in % of price)
DISTANCE_BUCKETS = [
    (0, 0.25),
    (0.25, 0.5),
    (0.5, 0.75),
    (0.75, 1.0),
    (1.0, 1.5),
    (1.5, 2.0),
    (2.0, 3.0),
    (3.0, 5.0),
    (5.0, float('inf'))
]

def get_distance_bucket(distance_pct):
    """Return which bucket a distance percentage falls into"""
    for bucket_min, bucket_max in DISTANCE_BUCKETS:
        if bucket_min <= distance_pct < bucket_max:
            return f"{bucket_min:.2f}-{bucket_max:.2f}%" if bucket_max != float('inf') else f"{bucket_min:.2f}%+"
    return "Unknown"

# Initialize tracking structure
# stats[level_name][context][distance_bucket] = {
#     'total_days': count,
#     'retraced_days': count,
#     'time_slots': {time_slot: count}
# }
stats = defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: {
    'total_days': 0,
    'retraced_days': 0,
    'time_slots': defaultdict(int)
})))

print("Processing hourly level retracement data...")
print(f"Tracking {len(HOURLY_LEVELS)} levels per day")
print(f"Distance buckets: {len(DISTANCE_BUCKETS)}")
print()

processed_count = 0

for date in dates[1:]:  # Start from second day
    processed_count += 1
    if processed_count % 500 == 0:
        print(f"Processed {processed_count} days...")
    
    # Map the NY trading day back to the actual date
    actual_date = date + pd.Timedelta(hours=7)
    
    # Define session time ranges
    asian_start = actual_date.replace(hour=20, minute=0, second=0) - pd.Timedelta(days=1)
    asian_end = actual_date.replace(hour=2, minute=0, second=0)
    london_start = asian_end
    london_end = actual_date.replace(hour=8, minute=0, second=0)
    ny_futures_start = london_end  # 8am
    ny_futures_end = actual_date.replace(hour=16, minute=0, second=0)  # 4pm
    
    # Extract session data
    asian_data = df.loc[asian_start:asian_end]
    london_data = df.loc[london_start:london_end]
    ny_futures_data = df.loc[ny_futures_start:ny_futures_end]
    
    # Skip if any session data is missing
    if asian_data.empty or london_data.empty or ny_futures_data.empty:
        continue
    
    # Get 8am open price (for distance calculations)
    open_8am_data = df.loc[ny_futures_start:ny_futures_start + pd.Timedelta(minutes=1)]
    if open_8am_data.empty:
        continue
    open_8am = open_8am_data['Open'].iloc[0]
    
    # Calculate London midpoint (2am-8am range)
    london_high = london_data['High'].max()
    london_low = london_data['Low'].min()
    if pd.isna(london_high) or pd.isna(london_low):
        continue
    london_midpoint = (london_high + london_low) / 2
    
    # Determine context: 8am opens above or below London midpoint
    position_context = 'above_london_mid' if open_8am > london_midpoint else 'below_london_mid'
    
    # Determine pattern (for additional filtering)
    asian_high = asian_data['High'].max()
    asian_low = asian_data['Low'].min()
    
    if pd.isna([asian_high, asian_low]).any():
        continue
    
    current_pattern = 'none'
    if london_high > asian_high and london_low < asian_low:
        current_pattern = 'london_engulfs_asia'
    elif london_high <= asian_high and london_low >= asian_low:
        current_pattern = 'asia_engulfs_london'
    elif london_high > asian_high and london_low >= asian_low:
        current_pattern = 'london_partial_up'
    elif london_low < asian_low and london_high <= asian_high:
        current_pattern = 'london_partial_down'
    
    # Now process each hourly level
    for hour, level_type, day_type in HOURLY_LEVELS:
        # Determine the timestamp for this level
        if day_type == 'prev_day':
            level_time = actual_date.replace(hour=hour, minute=0, second=0) - pd.Timedelta(days=1)
        else:
            level_time = actual_date.replace(hour=hour, minute=0, second=0)
        
        # Calculate the level price
        if level_type == 'open':
            # Get the open at this hour
            level_data = df.loc[level_time:level_time + pd.Timedelta(minutes=1)]
            if level_data.empty:
                continue
            level_price = level_data['Open'].iloc[0]
        else:  # midpoint
            # Get the hourly range and calculate midpoint
            hour_end = level_time + pd.Timedelta(hours=1)
            hour_data = df.loc[level_time:hour_end]
            if hour_data.empty:
                continue
            hour_high = hour_data['High'].max()
            hour_low = hour_data['Low'].min()
            if pd.isna(hour_high) or pd.isna(hour_low):
                continue
            level_price = (hour_high + hour_low) / 2
        
        # Calculate distance from 8am open to this level (as % of price)
        distance_points = abs(open_8am - level_price)
        distance_pct = (distance_points / open_8am) * 100
        distance_bucket = get_distance_bucket(distance_pct)
        
        # Create level name
        hour_12 = hour % 12
        if hour_12 == 0:
            hour_12 = 12
        am_pm = 'pm' if hour >= 12 else 'am'
        level_name = f"{hour_12:02d}:00{am_pm}_{level_type}"
        
        # Check if this level was retraced during NY futures session (8am-4pm)
        was_retraced = False
        retracement_time = None
        
        for idx, row in ny_futures_data.iterrows():
            if row['Low'] <= level_price <= row['High']:
                was_retraced = True
                retracement_time = idx
                break
        
        # Update statistics for different contexts
        contexts_to_update = [
            'overall',
            position_context,
            current_pattern,
            f"{position_context}_{current_pattern}"
        ]
        
        for context in contexts_to_update:
            stats[level_name][context][distance_bucket]['total_days'] += 1
            
            if was_retraced:
                stats[level_name][context][distance_bucket]['retraced_days'] += 1
                
                # Track timing (15-minute slots)
                hour = retracement_time.hour
                minute = retracement_time.minute
                slot_minute = (minute // 15) * 15
                time_slot = f"{hour:02d}:{slot_minute:02d}"
                stats[level_name][context][distance_bucket]['time_slots'][time_slot] += 1

print(f"\nProcessed {processed_count} days total")
print("\n" + "=" * 120)
print("ANALYSIS COMPLETE - Generating Reports")
print("=" * 120)

# ========================================================================
# DISPLAY FUNCTIONS
# ========================================================================

def display_level_summary(level_name, level_stats):
    """Display summary statistics for a single level across all contexts"""
    print("\n" + "=" * 120)
    print(f"LEVEL: {level_name}")
    print("=" * 120)
    
    # Overall stats first
    overall_stats = level_stats['overall']
    
    print("\nOVERALL STATISTICS (all days combined):")
    print("-" * 120)
    
    total_overall = sum(bucket_data['total_days'] for bucket_data in overall_stats.values())
    retraced_overall = sum(bucket_data['retraced_days'] for bucket_data in overall_stats.values())
    
    if total_overall > 0:
        overall_pct = (retraced_overall / total_overall) * 100
        print(f"Total days: {total_overall} | Retraced: {retraced_overall} | Rate: {overall_pct:.2f}%")
        
        print("\nBy Distance Bucket:")
        
        # Build table data
        bucket_rows = []
        for bucket_min, bucket_max in DISTANCE_BUCKETS:
            bucket_name = f"{bucket_min:.2f}-{bucket_max:.2f}%" if bucket_max != float('inf') else f"{bucket_min:.2f}%+"
            
            if bucket_name in overall_stats:
                data = overall_stats[bucket_name]
                if data['total_days'] > 0:
                    rate = (data['retraced_days'] / data['total_days']) * 100
                    bucket_rows.append({
                        'Distance_Bucket': bucket_name,
                        'Total': data['total_days'],
                        'Retraced': data['retraced_days'],
                        'Rate_%': round(rate, 2)
                    })
        
        if bucket_rows:
            bucket_df = pd.DataFrame(bucket_rows)
            print(bucket_df.to_string(index=False))
    
    # Position context stats
    print("\n\nBY POSITION RELATIVE TO LONDON MIDPOINT:")
    print("-" * 120)
    
    position_rows = []
    for position in ['above_london_mid', 'below_london_mid']:
        position_stats = level_stats[position]
        total = sum(bucket_data['total_days'] for bucket_data in position_stats.values())
        retraced = sum(bucket_data['retraced_days'] for bucket_data in position_stats.values())
        
        if total > 0:
            rate = (retraced / total) * 100
            position_label = "8am ABOVE London mid" if position == 'above_london_mid' else "8am BELOW London mid"
            position_rows.append({
                'Position': position_label,
                'Total': total,
                'Retraced': retraced,
                'Rate_%': round(rate, 2)
            })
    
    if position_rows:
        position_df = pd.DataFrame(position_rows)
        print(position_df.to_string(index=False))
    
    # Pattern context stats
    print("\n\nBY SESSION PATTERN:")
    print("-" * 120)
    
    patterns = ['london_engulfs_asia', 'asia_engulfs_london', 'london_partial_up', 'london_partial_down']
    pattern_labels = {
        'london_engulfs_asia': 'London Engulfs Asia',
        'asia_engulfs_london': 'Asia Engulfs London',
        'london_partial_up': 'London Partial Engulf Up',
        'london_partial_down': 'London Partial Engulf Down'
    }
    
    pattern_rows = []
    for pattern in patterns:
        pattern_stats = level_stats[pattern]
        total = sum(bucket_data['total_days'] for bucket_data in pattern_stats.values())
        retraced = sum(bucket_data['retraced_days'] for bucket_data in pattern_stats.values())
        
        if total > 0:
            rate = (retraced / total) * 100
            pattern_rows.append({
                'Pattern': pattern_labels[pattern],
                'Total': total,
                'Retraced': retraced,
                'Rate_%': round(rate, 2)
            })
    
    if pattern_rows:
        pattern_df = pd.DataFrame(pattern_rows)
        print(pattern_df.to_string(index=False))


def display_timing_analysis(level_name, level_stats):
    """Display timing analysis for when retracements occur"""
    print("\n" + "=" * 120)
    print(f"TIMING ANALYSIS: {level_name}")
    print("When during 8am-4pm session does retracement occur?")
    print("=" * 120)
    
    # Aggregate all timing data across all buckets for overall context
    overall_timing = defaultdict(int)
    total_retracements = 0
    
    for bucket_data in level_stats['overall'].values():
        for time_slot, count in bucket_data['time_slots'].items():
            overall_timing[time_slot] += count
            total_retracements += count
    
    if total_retracements > 0:
        print(f"\nTotal retracements tracked: {total_retracements}")
        
        # Build timing table
        timing_rows = []
        sorted_slots = sorted(overall_timing.items())
        for time_slot, count in sorted_slots:
            pct = (count / total_retracements) * 100
            timing_rows.append({
                'Time_Slot': time_slot,
                'Count': count,
                'Pct_of_Total': round(pct, 2)
            })
        
        timing_df = pd.DataFrame(timing_rows)
        print("\n" + timing_df.to_string(index=False))


# ========================================================================
# EXPORT TO CSV
# ========================================================================

export_data = []

for level_name, level_stats in stats.items():
    for context, context_stats in level_stats.items():
        for distance_bucket, bucket_data in context_stats.items():
            if bucket_data['total_days'] > 0:
                rate = (bucket_data['retraced_days'] / bucket_data['total_days']) * 100
                
                export_data.append({
                    'Level': level_name,
                    'Context': context,
                    'Distance_Bucket': distance_bucket,
                    'Total_Days': bucket_data['total_days'],
                    'Retraced_Days': bucket_data['retraced_days'],
                    'Retracement_Rate_Pct': rate
                })

export_df = pd.DataFrame(export_data)
output_path = r"C:\Users\tammm\NQ and ES analysis\hourly_level_retracements.csv"
export_df.to_csv(output_path, index=False)
print(f"\nDetailed data exported to: {output_path}")

# ========================================================================
# GENERATE SUMMARY REPORT
# ========================================================================

print("\n\n")
print("=" * 120)
print("TOP PERFORMING LEVELS - OVERALL")
print("=" * 120)

# Calculate overall stats for each level and sort by retracement rate
level_summary = []
for level_name, level_stats in stats.items():
    overall_stats = level_stats['overall']
    total = sum(bucket_data['total_days'] for bucket_data in overall_stats.values())
    retraced = sum(bucket_data['retraced_days'] for bucket_data in overall_stats.values())
    
    if total > 0:
        rate = (retraced / total) * 100
        level_summary.append({
            'Level': level_name,
            'Total_Days': total,
            'Retraced': retraced,
            'Rate_%': round(rate, 2)
        })

# Sort by retracement rate
level_summary_sorted = sorted(level_summary, key=lambda x: x['Rate_%'], reverse=True)

# Create DataFrame for top 20
top_20_df = pd.DataFrame(level_summary_sorted[:20])
top_20_df.insert(0, 'Rank', range(1, len(top_20_df) + 1))
print("\n" + top_20_df.to_string(index=False))

# ========================================================================
# DETAILED REPORTS FOR TOP LEVELS
# ========================================================================

print("\n\n")
print("=" * 120)
print("DETAILED ANALYSIS - TOP 5 LEVELS")
print("=" * 120)

for i in range(min(5, len(level_summary_sorted))):
    item = level_summary_sorted[i]
    display_level_summary(item['Level'], stats[item['Level']])
    display_timing_analysis(item['Level'], stats[item['Level']])

print("\n\n" + "=" * 120)
print("ANALYSIS COMPLETE")
print("=" * 120)
print(f"\nFull dataset exported to: {output_path}")
print("Use the CSV for deeper analysis, filtering, and visualization")

Processing hourly level retracement data...
Tracking 27 levels per day
Distance buckets: 9

Processed 500 days...
Processed 1000 days...
Processed 1500 days...
Processed 2000 days...
Processed 2500 days...
Processed 3000 days...
Processed 3500 days...

Processed 3785 days total

ANALYSIS COMPLETE - Generating Reports

Detailed data exported to: C:\Users\tammm\NQ and ES analysis\hourly_level_retracements.csv



TOP PERFORMING LEVELS - OVERALL

 Rank            Level  Total_Days  Retraced  Rate_%
    1     07:00am_open        3127      2766   88.46
    2 06:00am_midpoint        3130      2731   87.25
    3     06:00am_open        3125      2638   84.42
    4 05:00am_midpoint        3130      2640   84.35
    5     05:00am_open        3124      2584   82.71
    6 04:00am_midpoint        3130      2558   81.73
    7     04:00am_open        3129      2490   79.58
    8 03:00am_midpoint        3130      2468   78.85
    9     03:00am_open        3128      2413   77.14
   10 02:00am_midpoint 

In [3]:
## Hourly Level Retracement Analysis - NY Futures Session (8am-4pm EST)
## Tracks: hourly opens & midpoints from 6pm-7am, distance buckets, timing, London midpoint context


# Define hourly levels to track (6pm previous day through 7am current day)
# Format: (hour, level_type) where hour is 0-23, level_type is 'open' or 'midpoint'
HOURLY_LEVELS = []

# Previous day: 18:00 (6pm) through 23:00 (11pm)
for hour in range(18, 24):
    HOURLY_LEVELS.append((hour, 'open', 'prev_day'))
    HOURLY_LEVELS.append((hour, 'midpoint', 'prev_day'))

# Current day: 00:00 (midnight) through 07:00 (7am)
for hour in range(0, 8):
    HOURLY_LEVELS.append((hour, 'open', 'current_day'))
    if hour < 7:  # 7-8am midpoint is included, but not 8am open
        HOURLY_LEVELS.append((hour, 'midpoint', 'current_day'))

# Distance buckets (in % of price)
DISTANCE_BUCKETS = [
    (0, 0.25),
    (0.25, 0.5),
    (0.5, 0.75),
    (0.75, 1.0),
    (1.0, 1.5),
    (1.5, 2.0),
    (2.0, 3.0),
    (3.0, 5.0),
    (5.0, float('inf'))
]

def get_distance_bucket(distance_pct):
    """Return which bucket a distance percentage falls into"""
    for bucket_min, bucket_max in DISTANCE_BUCKETS:
        if bucket_min <= distance_pct < bucket_max:
            return f"{bucket_min:.2f}-{bucket_max:.2f}%" if bucket_max != float('inf') else f"{bucket_min:.2f}%+"
    return "Unknown"

# Initialize tracking structure
# stats[level_name][context][distance_bucket] = {
#     'total_days': count,
#     'retraced_days': count,
#     'time_slots': {time_slot: count}
# }
stats = defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: {
    'total_days': 0,
    'retraced_days': 0,
    'time_slots': defaultdict(int)
})))

print("Processing hourly level retracement data...")
print(f"Tracking {len(HOURLY_LEVELS)} levels per day")
print(f"Distance buckets: {len(DISTANCE_BUCKETS)}")
print()

processed_count = 0

for date in dates[1:]:  # Start from second day
    processed_count += 1
    if processed_count % 500 == 0:
        print(f"Processed {processed_count} days...")
    
    # Map the NY trading day back to the actual date
    actual_date = date + pd.Timedelta(hours=7)
    
    # Define session time ranges
    asian_start = actual_date.replace(hour=20, minute=0, second=0) - pd.Timedelta(days=1)
    asian_end = actual_date.replace(hour=2, minute=0, second=0)
    london_start = asian_end
    london_end = actual_date.replace(hour=8, minute=0, second=0)
    ny_futures_start = london_end  # 8am
    ny_futures_end = actual_date.replace(hour=16, minute=0, second=0)  # 4pm
    
    # Extract session data
    asian_data = df.loc[asian_start:asian_end]
    london_data = df.loc[london_start:london_end]
    ny_futures_data = df.loc[ny_futures_start:ny_futures_end]
    
    # Skip if any session data is missing
    if asian_data.empty or london_data.empty or ny_futures_data.empty:
        continue
    
    # Get 8am open price (for distance calculations)
    open_8am_data = df.loc[ny_futures_start:ny_futures_start + pd.Timedelta(minutes=1)]
    if open_8am_data.empty:
        continue
    open_8am = open_8am_data['Open'].iloc[0]
    
    # Calculate London midpoint (2am-8am range)
    london_high = london_data['High'].max()
    london_low = london_data['Low'].min()
    if pd.isna(london_high) or pd.isna(london_low):
        continue
    london_midpoint = (london_high + london_low) / 2
    
    # Determine pattern (for additional filtering)
    asian_high = asian_data['High'].max()
    asian_low = asian_data['Low'].min()
    
    if pd.isna([asian_high, asian_low]).any():
        continue
    
    current_pattern = 'none'
    if london_high > asian_high and london_low < asian_low:
        current_pattern = 'london_engulfs_asia'
    elif london_high <= asian_high and london_low >= asian_low:
        current_pattern = 'asia_engulfs_london'
    elif london_high > asian_high and london_low >= asian_low:
        current_pattern = 'london_partial_up'
    elif london_low < asian_low and london_high <= asian_high:
        current_pattern = 'london_partial_down'
    
    # Now process each hourly level
    for hour, level_type, day_type in HOURLY_LEVELS:
        # Determine the timestamp for this level
        if day_type == 'prev_day':
            level_time = actual_date.replace(hour=hour, minute=0, second=0) - pd.Timedelta(days=1)
        else:
            level_time = actual_date.replace(hour=hour, minute=0, second=0)
        
        # Calculate the level price
        if level_type == 'open':
            # Get the open at this hour
            level_data = df.loc[level_time:level_time + pd.Timedelta(minutes=1)]
            if level_data.empty:
                continue
            level_price = level_data['Open'].iloc[0]
        else:  # midpoint
            # Get the hourly range and calculate midpoint
            hour_end = level_time + pd.Timedelta(hours=1)
            hour_data = df.loc[level_time:hour_end]
            if hour_data.empty:
                continue
            hour_high = hour_data['High'].max()
            hour_low = hour_data['Low'].min()
            if pd.isna(hour_high) or pd.isna(hour_low):
                continue
            level_price = (hour_high + hour_low) / 2
        
        # Calculate distance from 8am open to this level (as % of price)
        distance_points = abs(open_8am - level_price)
        distance_pct = (distance_points / open_8am) * 100
        distance_bucket = get_distance_bucket(distance_pct)
        
        # Create level name
        hour_12 = hour % 12
        if hour_12 == 0:
            hour_12 = 12
        am_pm = 'pm' if hour >= 12 else 'am'
        level_name = f"{hour_12:02d}:00{am_pm}_{level_type}"
        
        # Check if this level was retraced during NY futures session (8am-4pm)
        was_retraced = False
        retracement_time = None
        
        for idx, row in ny_futures_data.iterrows():
            if row['Low'] <= level_price <= row['High']:
                was_retraced = True
                retracement_time = idx
                break
        
        # Update statistics for different contexts
        contexts_to_update = [
            'overall',
            current_pattern,
            f"{current_pattern}_{distance_bucket}"  # Pattern + distance combination
        ]
        
        for context in contexts_to_update:
            stats[level_name][context][distance_bucket]['total_days'] += 1
            
            if was_retraced:
                stats[level_name][context][distance_bucket]['retraced_days'] += 1
                
                # Track timing (15-minute slots)
                hour = retracement_time.hour
                minute = retracement_time.minute
                slot_minute = (minute // 15) * 15
                time_slot = f"{hour:02d}:{slot_minute:02d}"
                stats[level_name][context][distance_bucket]['time_slots'][time_slot] += 1

print(f"\nProcessed {processed_count} days total")
print("\n" + "=" * 120)
print("ANALYSIS COMPLETE - Generating Reports")
print("=" * 120)

# ========================================================================
# DISPLAY FUNCTIONS
# ========================================================================

def display_level_summary(level_name, level_stats):
    """Display summary statistics for a single level across all contexts"""
    print("\n" + "=" * 120)
    print(f"LEVEL: {level_name}")
    print("=" * 120)
    
    # Overall stats first
    overall_stats = level_stats['overall']
    
    print("\nOVERALL STATISTICS (all days combined):")
    print("-" * 120)
    
    total_overall = sum(bucket_data['total_days'] for bucket_data in overall_stats.values())
    retraced_overall = sum(bucket_data['retraced_days'] for bucket_data in overall_stats.values())
    
    if total_overall > 0:
        overall_pct = (retraced_overall / total_overall) * 100
        print(f"Total days: {total_overall} | Retraced: {retraced_overall} | Rate: {overall_pct:.2f}%")
        
        print("\nBy Distance Bucket:")
        
        # Build table data
        bucket_rows = []
        for bucket_min, bucket_max in DISTANCE_BUCKETS:
            bucket_name = f"{bucket_min:.2f}-{bucket_max:.2f}%" if bucket_max != float('inf') else f"{bucket_min:.2f}%+"
            
            if bucket_name in overall_stats:
                data = overall_stats[bucket_name]
                if data['total_days'] > 0:
                    rate = (data['retraced_days'] / data['total_days']) * 100
                    bucket_rows.append({
                        'Distance_Bucket': bucket_name,
                        'Total': data['total_days'],
                        'Retraced': data['retraced_days'],
                        'Rate_%': round(rate, 2)
                    })
        
        if bucket_rows:
            bucket_df = pd.DataFrame(bucket_rows)
            print(bucket_df.to_string(index=False))
    
    # Pattern context stats
    print("\n\nBY SESSION PATTERN:")
    print("-" * 120)
    
    patterns = ['london_engulfs_asia', 'asia_engulfs_london', 'london_partial_up', 'london_partial_down']
    pattern_labels = {
        'london_engulfs_asia': 'London Engulfs Asia',
        'asia_engulfs_london': 'Asia Engulfs London',
        'london_partial_up': 'London Partial Engulf Up',
        'london_partial_down': 'London Partial Engulf Down'
    }
    
    pattern_rows = []
    for pattern in patterns:
        pattern_stats = level_stats[pattern]
        total = sum(bucket_data['total_days'] for bucket_data in pattern_stats.values())
        retraced = sum(bucket_data['retraced_days'] for bucket_data in pattern_stats.values())
        
        if total > 0:
            rate = (retraced / total) * 100
            pattern_rows.append({
                'Pattern': pattern_labels[pattern],
                'Total': total,
                'Retraced': retraced,
                'Rate_%': round(rate, 2)
            })
    
    if pattern_rows:
        pattern_df = pd.DataFrame(pattern_rows)
        print(pattern_df.to_string(index=False))
    
    # Pattern × Distance combinations
    print("\n\nPATTERN × DISTANCE COMBINATIONS (showing combinations with 20+ samples):")
    print("-" * 120)
    
    combo_rows = []
    for pattern in patterns:
        for bucket_min, bucket_max in DISTANCE_BUCKETS:
            bucket_name = f"{bucket_min:.2f}-{bucket_max:.2f}%" if bucket_max != float('inf') else f"{bucket_min:.2f}%+"
            combo_key = f"{pattern}_{bucket_name}"
            
            if combo_key in level_stats:
                combo_stats = level_stats[combo_key]
                total = sum(bucket_data['total_days'] for bucket_data in combo_stats.values())
                retraced = sum(bucket_data['retraced_days'] for bucket_data in combo_stats.values())
                
                if total >= 20:  # Only show combinations with sufficient sample size
                    rate = (retraced / total) * 100
                    combo_rows.append({
                        'Pattern': pattern_labels[pattern],
                        'Distance': bucket_name,
                        'Total': total,
                        'Retraced': retraced,
                        'Rate_%': round(rate, 2)
                    })
    
    if combo_rows:
        # Sort by rate descending
        combo_df = pd.DataFrame(combo_rows)
        combo_df = combo_df.sort_values('Rate_%', ascending=False)
        print(combo_df.to_string(index=False))
    else:
        print("No pattern-distance combinations with 20+ samples found.")


def display_timing_analysis(level_name, level_stats):
    """Display timing analysis for when retracements occur"""
    print("\n" + "=" * 120)
    print(f"TIMING ANALYSIS: {level_name}")
    print("When during 8am-4pm session does retracement occur?")
    print("=" * 120)
    
    # Aggregate all timing data across all buckets for overall context
    overall_timing = defaultdict(int)
    total_retracements = 0
    
    for bucket_data in level_stats['overall'].values():
        for time_slot, count in bucket_data['time_slots'].items():
            overall_timing[time_slot] += count
            total_retracements += count
    
    if total_retracements > 0:
        print(f"\nTotal retracements tracked: {total_retracements}")
        
        # Build timing table
        timing_rows = []
        sorted_slots = sorted(overall_timing.items())
        for time_slot, count in sorted_slots:
            pct = (count / total_retracements) * 100
            timing_rows.append({
                'Time_Slot': time_slot,
                'Count': count,
                'Pct_of_Total': round(pct, 2)
            })
        
        timing_df = pd.DataFrame(timing_rows)
        print("\n" + timing_df.to_string(index=False))


# ========================================================================
# EXPORT TO CSV
# ========================================================================

export_data = []

for level_name, level_stats in stats.items():
    for context, context_stats in level_stats.items():
        for distance_bucket, bucket_data in context_stats.items():
            if bucket_data['total_days'] > 0:
                rate = (bucket_data['retraced_days'] / bucket_data['total_days']) * 100
                
                export_data.append({
                    'Level': level_name,
                    'Context': context,
                    'Distance_Bucket': distance_bucket,
                    'Total_Days': bucket_data['total_days'],
                    'Retraced_Days': bucket_data['retraced_days'],
                    'Retracement_Rate_Pct': rate
                })

export_df = pd.DataFrame(export_data)
output_path = r"C:\Users\tammm\NQ and ES analysis\hourly_level_retracements.csv"
export_df.to_csv(output_path, index=False)
print(f"\nDetailed data exported to: {output_path}")

# ========================================================================
# GENERATE SUMMARY REPORT
# ========================================================================

print("\n\n")
print("=" * 120)
print("TOP PERFORMING LEVELS - OVERALL")
print("=" * 120)

# Calculate overall stats for each level and sort by retracement rate
level_summary = []
for level_name, level_stats in stats.items():
    overall_stats = level_stats['overall']
    total = sum(bucket_data['total_days'] for bucket_data in overall_stats.values())
    retraced = sum(bucket_data['retraced_days'] for bucket_data in overall_stats.values())
    
    if total > 0:
        rate = (retraced / total) * 100
        level_summary.append({
            'Level': level_name,
            'Total_Days': total,
            'Retraced': retraced,
            'Rate_%': round(rate, 2)
        })

# Sort by retracement rate
level_summary_sorted = sorted(level_summary, key=lambda x: x['Rate_%'], reverse=True)

# Create DataFrame for top 20
top_20_df = pd.DataFrame(level_summary_sorted[:20])
top_20_df.insert(0, 'Rank', range(1, len(top_20_df) + 1))
print("\n" + top_20_df.to_string(index=False))

# ========================================================================
# DETAILED REPORTS FOR TOP LEVELS
# ========================================================================

print("\n\n")
print("=" * 120)
print("DETAILED ANALYSIS - TOP 5 LEVELS")
print("=" * 120)

for i in range(min(5, len(level_summary_sorted))):
    item = level_summary_sorted[i]
    display_level_summary(item['Level'], stats[item['Level']])
    display_timing_analysis(item['Level'], stats[item['Level']])

print("\n\n" + "=" * 120)
print("ANALYSIS COMPLETE")
print("=" * 120)
print(f"\nFull dataset exported to: {output_path}")
print("Use the CSV for deeper analysis, filtering, and visualization")

Processing hourly level retracement data...
Tracking 27 levels per day
Distance buckets: 9

Processed 500 days...
Processed 1000 days...
Processed 1500 days...
Processed 2000 days...
Processed 2500 days...
Processed 3000 days...
Processed 3500 days...

Processed 3785 days total

ANALYSIS COMPLETE - Generating Reports

Detailed data exported to: C:\Users\tammm\NQ and ES analysis\hourly_level_retracements.csv



TOP PERFORMING LEVELS - OVERALL

 Rank            Level  Total_Days  Retraced  Rate_%
    1     07:00am_open        3127      2766   88.46
    2 06:00am_midpoint        3130      2731   87.25
    3     06:00am_open        3125      2638   84.42
    4 05:00am_midpoint        3130      2640   84.35
    5     05:00am_open        3124      2584   82.71
    6 04:00am_midpoint        3130      2558   81.73
    7     04:00am_open        3129      2490   79.58
    8 03:00am_midpoint        3130      2468   78.85
    9     03:00am_open        3128      2413   77.14
   10 02:00am_midpoint 

In [4]:
## Hourly Level Retracement Analysis - NY Futures Session (8am-4pm EST)
## Tracks: hourly opens & midpoints from 6pm-7am, distance buckets, timing, London midpoint context

# Define hourly levels to track (6pm previous day through 7am current day)
# Format: (hour, level_type) where hour is 0-23, level_type is 'open' or 'midpoint'
HOURLY_LEVELS = []

# Previous day: 18:00 (6pm) through 23:00 (11pm)
for hour in range(18, 24):
    HOURLY_LEVELS.append((hour, 'open', 'prev_day'))
    HOURLY_LEVELS.append((hour, 'midpoint', 'prev_day'))

# Current day: 00:00 (midnight) through 07:00 (7am)
for hour in range(0, 8):
    HOURLY_LEVELS.append((hour, 'open', 'current_day'))
    if hour < 7:  # 7-8am midpoint is included, but not 8am open
        HOURLY_LEVELS.append((hour, 'midpoint', 'current_day'))

# Distance buckets (in % of price)
DISTANCE_BUCKETS = [
    (0, 0.25),
    (0.25, 0.5),
    (0.5, 0.75),
    (0.75, 1.0),
    (1.0, 1.5),
    (1.5, 2.0),
    (2.0, 3.0),
    (3.0, 5.0),
    (5.0, float('inf'))
]

def get_distance_bucket(distance_pct):
    """Return which bucket a distance percentage falls into"""
    for bucket_min, bucket_max in DISTANCE_BUCKETS:
        if bucket_min <= distance_pct < bucket_max:
            return f"{bucket_min:.2f}-{bucket_max:.2f}%" if bucket_max != float('inf') else f"{bucket_min:.2f}%+"
    return "Unknown"

# Initialize tracking structure
# stats[level_name][context][distance_bucket] = {
#     'total_days': count,
#     'retraced_days': count,
#     'time_slots': {time_slot: count}
# }
stats = defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: {
    'total_days': 0,
    'retraced_days': 0,
    'time_slots': defaultdict(int)
})))

print("Processing hourly level retracement data...")
print(f"Tracking {len(HOURLY_LEVELS)} levels per day")
print(f"Distance buckets: {len(DISTANCE_BUCKETS)}")
print()

processed_count = 0

for date in dates[1:]:  # Start from second day
    processed_count += 1
    if processed_count % 500 == 0:
        print(f"Processed {processed_count} days...")
    
    # Map the NY trading day back to the actual date
    actual_date = date + pd.Timedelta(hours=7)
    
    # Define session time ranges
    asian_start = actual_date.replace(hour=20, minute=0, second=0) - pd.Timedelta(days=1)
    asian_end = actual_date.replace(hour=2, minute=0, second=0)
    london_start = asian_end
    london_end = actual_date.replace(hour=8, minute=0, second=0)
    ny_futures_start = london_end  # 8am
    ny_futures_end = actual_date.replace(hour=16, minute=0, second=0)  # 4pm
    
    # Extract session data
    asian_data = df.loc[asian_start:asian_end]
    london_data = df.loc[london_start:london_end]
    ny_futures_data = df.loc[ny_futures_start:ny_futures_end]
    
    # Skip if any session data is missing
    if asian_data.empty or london_data.empty or ny_futures_data.empty:
        continue
    
    # Get 8am open price (for distance calculations)
    open_8am_data = df.loc[ny_futures_start:ny_futures_start + pd.Timedelta(minutes=1)]
    if open_8am_data.empty:
        continue
    open_8am = open_8am_data['Open'].iloc[0]
    
    # Calculate London midpoint (2am-8am range)
    london_high = london_data['High'].max()
    london_low = london_data['Low'].min()
    if pd.isna(london_high) or pd.isna(london_low):
        continue
    london_midpoint = (london_high + london_low) / 2
    
    # Determine pattern (for additional filtering)
    asian_high = asian_data['High'].max()
    asian_low = asian_data['Low'].min()
    
    if pd.isna([asian_high, asian_low]).any():
        continue
    
    current_pattern = 'none'
    if london_high > asian_high and london_low < asian_low:
        current_pattern = 'london_engulfs_asia'
    elif london_high <= asian_high and london_low >= asian_low:
        current_pattern = 'asia_engulfs_london'
    elif london_high > asian_high and london_low >= asian_low:
        current_pattern = 'london_partial_up'
    elif london_low < asian_low and london_high <= asian_high:
        current_pattern = 'london_partial_down'
    
    # Now process each hourly level
    for hour, level_type, day_type in HOURLY_LEVELS:
        # Determine the timestamp for this level
        if day_type == 'prev_day':
            level_time = actual_date.replace(hour=hour, minute=0, second=0) - pd.Timedelta(days=1)
        else:
            level_time = actual_date.replace(hour=hour, minute=0, second=0)
        
        # Calculate the level price
        if level_type == 'open':
            # Get the open at this hour
            level_data = df.loc[level_time:level_time + pd.Timedelta(minutes=1)]
            if level_data.empty:
                continue
            level_price = level_data['Open'].iloc[0]
        else:  # midpoint
            # Get the hourly range and calculate midpoint
            hour_end = level_time + pd.Timedelta(hours=1)
            hour_data = df.loc[level_time:hour_end]
            if hour_data.empty:
                continue
            hour_high = hour_data['High'].max()
            hour_low = hour_data['Low'].min()
            if pd.isna(hour_high) or pd.isna(hour_low):
                continue
            level_price = (hour_high + hour_low) / 2
        
        # Calculate distance from 8am open to this level (as % of price)
        distance_points = abs(open_8am - level_price)
        distance_pct = (distance_points / open_8am) * 100
        distance_bucket = get_distance_bucket(distance_pct)
        
        # Create level name
        hour_12 = hour % 12
        if hour_12 == 0:
            hour_12 = 12
        am_pm = 'pm' if hour >= 12 else 'am'
        level_name = f"{hour_12:02d}:00{am_pm}_{level_type}"
        
        # Check if this level was retraced during NY futures session (8am-4pm)
        was_retraced = False
        retracement_time = None
        
        for idx, row in ny_futures_data.iterrows():
            if row['Low'] <= level_price <= row['High']:
                was_retraced = True
                retracement_time = idx
                break
        
        # Update statistics for different contexts
        contexts_to_update = [
            'overall',
            current_pattern,
            f"{current_pattern}_{distance_bucket}"  # Pattern + distance combination
        ]
        
        for context in contexts_to_update:
            stats[level_name][context][distance_bucket]['total_days'] += 1
            
            if was_retraced:
                stats[level_name][context][distance_bucket]['retraced_days'] += 1
                
                # Track timing (15-minute slots)
                hour = retracement_time.hour
                minute = retracement_time.minute
                slot_minute = (minute // 15) * 15
                time_slot = f"{hour:02d}:{slot_minute:02d}"
                stats[level_name][context][distance_bucket]['time_slots'][time_slot] += 1

print(f"\nProcessed {processed_count} days total")
print("\n" + "=" * 120)
print("ANALYSIS COMPLETE - Generating Reports")
print("=" * 120)

# ========================================================================
# DISPLAY FUNCTIONS
# ========================================================================

def display_level_summary(level_name, level_stats):
    """Display summary statistics for a single level across all contexts"""
    print("\n" + "=" * 120)
    print(f"LEVEL: {level_name}")
    print("=" * 120)
    
    # Overall stats first
    overall_stats = level_stats['overall']
    
    print("\nOVERALL STATISTICS (all days combined):")
    print("-" * 120)
    
    total_overall = sum(bucket_data['total_days'] for bucket_data in overall_stats.values())
    retraced_overall = sum(bucket_data['retraced_days'] for bucket_data in overall_stats.values())
    
    if total_overall > 0:
        overall_pct = (retraced_overall / total_overall) * 100
        print(f"Total days: {total_overall} | Retraced: {retraced_overall} | Rate: {overall_pct:.2f}%")
        
        print("\nBy Distance Bucket:")
        
        # Build table data
        bucket_rows = []
        for bucket_min, bucket_max in DISTANCE_BUCKETS:
            bucket_name = f"{bucket_min:.2f}-{bucket_max:.2f}%" if bucket_max != float('inf') else f"{bucket_min:.2f}%+"
            
            if bucket_name in overall_stats:
                data = overall_stats[bucket_name]
                if data['total_days'] > 0:
                    rate = (data['retraced_days'] / data['total_days']) * 100
                    bucket_rows.append({
                        'Distance_Bucket': bucket_name,
                        'Total': data['total_days'],
                        'Retraced': data['retraced_days'],
                        'Rate_%': round(rate, 2)
                    })
        
        if bucket_rows:
            bucket_df = pd.DataFrame(bucket_rows)
            print(bucket_df.to_string(index=False))
    
    # Pattern context stats
    print("\n\nBY SESSION PATTERN:")
    print("-" * 120)
    
    patterns = ['london_engulfs_asia', 'asia_engulfs_london', 'london_partial_up', 'london_partial_down']
    pattern_labels = {
        'london_engulfs_asia': 'London Engulfs Asia',
        'asia_engulfs_london': 'Asia Engulfs London',
        'london_partial_up': 'London Partial Engulf Up',
        'london_partial_down': 'London Partial Engulf Down'
    }
    
    pattern_rows = []
    for pattern in patterns:
        pattern_stats = level_stats[pattern]
        total = sum(bucket_data['total_days'] for bucket_data in pattern_stats.values())
        retraced = sum(bucket_data['retraced_days'] for bucket_data in pattern_stats.values())
        
        if total > 0:
            rate = (retraced / total) * 100
            pattern_rows.append({
                'Pattern': pattern_labels[pattern],
                'Total': total,
                'Retraced': retraced,
                'Rate_%': round(rate, 2)
            })
    
    if pattern_rows:
        pattern_df = pd.DataFrame(pattern_rows)
        print(pattern_df.to_string(index=False))
    
    # Pattern × Distance combinations
    print("\n\nPATTERN × DISTANCE COMBINATIONS (showing combinations with 20+ samples):")
    print("-" * 120)
    
    combo_rows = []
    for pattern in patterns:
        for bucket_min, bucket_max in DISTANCE_BUCKETS:
            bucket_name = f"{bucket_min:.2f}-{bucket_max:.2f}%" if bucket_max != float('inf') else f"{bucket_min:.2f}%+"
            combo_key = f"{pattern}_{bucket_name}"
            
            if combo_key in level_stats:
                combo_stats = level_stats[combo_key]
                total = sum(bucket_data['total_days'] for bucket_data in combo_stats.values())
                retraced = sum(bucket_data['retraced_days'] for bucket_data in combo_stats.values())
                
                if total >= 20:  # Only show combinations with sufficient sample size
                    rate = (retraced / total) * 100
                    combo_rows.append({
                        'Pattern': pattern_labels[pattern],
                        'Distance': bucket_name,
                        'Total': total,
                        'Retraced': retraced,
                        'Rate_%': round(rate, 2)
                    })
    
    if combo_rows:
        # Sort by rate descending
        combo_df = pd.DataFrame(combo_rows)
        combo_df = combo_df.sort_values('Rate_%', ascending=False)
        print(combo_df.to_string(index=False))
    else:
        print("No pattern-distance combinations with 20+ samples found.")


def display_timing_analysis(level_name, level_stats):
    """Display timing analysis for when retracements occur"""
    print("\n" + "=" * 120)
    print(f"TIMING ANALYSIS: {level_name}")
    print("When during 8am-4pm session does retracement occur?")
    print("=" * 120)
    
    # Aggregate all timing data across all buckets for overall context
    overall_timing = defaultdict(int)
    total_retracements = 0
    
    for bucket_data in level_stats['overall'].values():
        for time_slot, count in bucket_data['time_slots'].items():
            overall_timing[time_slot] += count
            total_retracements += count
    
    if total_retracements > 0:
        print(f"\nTotal retracements tracked: {total_retracements}")
        
        # Build timing table
        timing_rows = []
        sorted_slots = sorted(overall_timing.items())
        for time_slot, count in sorted_slots:
            pct = (count / total_retracements) * 100
            timing_rows.append({
                'Time_Slot': time_slot,
                'Count': count,
                'Pct_of_Total': round(pct, 2)
            })
        
        timing_df = pd.DataFrame(timing_rows)
        print("\n" + timing_df.to_string(index=False))


# ========================================================================
# EXPORT TO CSV
# ========================================================================

export_data = []

for level_name, level_stats in stats.items():
    for context, context_stats in level_stats.items():
        for distance_bucket, bucket_data in context_stats.items():
            if bucket_data['total_days'] > 0:
                rate = (bucket_data['retraced_days'] / bucket_data['total_days']) * 100
                
                export_data.append({
                    'Level': level_name,
                    'Context': context,
                    'Distance_Bucket': distance_bucket,
                    'Total_Days': bucket_data['total_days'],
                    'Retraced_Days': bucket_data['retraced_days'],
                    'Retracement_Rate_Pct': rate
                })

export_df = pd.DataFrame(export_data)
output_path = r"C:\Users\tammm\NQ and ES analysis\hourly_level_retracements.csv"
export_df.to_csv(output_path, index=False)
print(f"\nDetailed data exported to: {output_path}")

# ========================================================================
# GENERATE SUMMARY REPORT
# ========================================================================

print("\n\n")
print("=" * 120)
print("TOP PERFORMING LEVELS - OVERALL")
print("=" * 120)

# Calculate overall stats for each level and sort by retracement rate
level_summary = []
for level_name, level_stats in stats.items():
    overall_stats = level_stats['overall']
    total = sum(bucket_data['total_days'] for bucket_data in overall_stats.values())
    retraced = sum(bucket_data['retraced_days'] for bucket_data in overall_stats.values())
    
    if total > 0:
        rate = (retraced / total) * 100
        level_summary.append({
            'Level': level_name,
            'Total_Days': total,
            'Retraced': retraced,
            'Rate_%': round(rate, 2)
        })

# Sort by retracement rate
level_summary_sorted = sorted(level_summary, key=lambda x: x['Rate_%'], reverse=True)

# Create DataFrame for top 20
top_20_df = pd.DataFrame(level_summary_sorted[:20])
top_20_df.insert(0, 'Rank', range(1, len(top_20_df) + 1))
print("\n" + top_20_df.to_string(index=False))

# ========================================================================
# COMPREHENSIVE ANALYSIS - ALL LEVELS IN TABLE FORMAT
# ========================================================================

print("\n\n")
print("=" * 120)
print("COMPREHENSIVE ANALYSIS - ALL LEVELS")
print("=" * 120)

# Sort levels by time order for easier reading
level_time_order = []
for hour in range(18, 24):  # 6pm-11pm
    level_time_order.append(f"{hour % 12 if hour % 12 != 0 else 12:02d}:00pm_open")
    level_time_order.append(f"{hour % 12 if hour % 12 != 0 else 12:02d}:00pm_midpoint")
for hour in range(0, 8):  # 12am-7am
    am_hour = hour if hour != 0 else 12
    level_time_order.append(f"{am_hour:02d}:00am_open")
    if hour < 7:
        level_time_order.append(f"{am_hour:02d}:00am_midpoint")

# 1. OVERALL RETRACEMENT RATES
print("\n1. OVERALL RETRACEMENT RATES")
print("-" * 120)

overall_data = []
for level_name in level_time_order:
    if level_name in stats:
        overall_stats = stats[level_name]['overall']
        total = sum(bucket_data['total_days'] for bucket_data in overall_stats.values())
        retraced = sum(bucket_data['retraced_days'] for bucket_data in overall_stats.values())
        
        if total > 0:
            rate = (retraced / total) * 100
            overall_data.append({
                'Level': level_name,
                'Total': total,
                'Retraced': retraced,
                'Rate_%': round(rate, 2)
            })

overall_df = pd.DataFrame(overall_data)
print(overall_df.to_string(index=False))

# 2. RETRACEMENT RATES BY DISTANCE BUCKET
print("\n\n2. RETRACEMENT RATES BY DISTANCE BUCKET")
print("-" * 120)

for bucket_min, bucket_max in DISTANCE_BUCKETS[:5]:  # Show first 5 buckets (most common)
    bucket_name = f"{bucket_min:.2f}-{bucket_max:.2f}%" if bucket_max != float('inf') else f"{bucket_min:.2f}%+"
    
    print(f"\nDistance: {bucket_name}")
    bucket_data = []
    
    for level_name in level_time_order:
        if level_name in stats and bucket_name in stats[level_name]['overall']:
            data = stats[level_name]['overall'][bucket_name]
            if data['total_days'] > 0:
                rate = (data['retraced_days'] / data['total_days']) * 100
                bucket_data.append({
                    'Level': level_name,
                    'Total': data['total_days'],
                    'Retraced': data['retraced_days'],
                    'Rate_%': round(rate, 2)
                })
    
    if bucket_data:
        bucket_df = pd.DataFrame(bucket_data)
        print(bucket_df.to_string(index=False))

# 3. RETRACEMENT RATES BY SESSION PATTERN
print("\n\n3. RETRACEMENT RATES BY SESSION PATTERN")
print("-" * 120)

patterns = ['london_engulfs_asia', 'asia_engulfs_london', 'london_partial_up', 'london_partial_down']
pattern_labels = {
    'london_engulfs_asia': 'London Engulfs Asia',
    'asia_engulfs_london': 'Asia Engulfs London',
    'london_partial_up': 'London Partial Up',
    'london_partial_down': 'London Partial Down'
}

for pattern in patterns:
    print(f"\nPattern: {pattern_labels[pattern]}")
    pattern_data = []
    
    for level_name in level_time_order:
        if level_name in stats and pattern in stats[level_name]:
            pattern_stats = stats[level_name][pattern]
            total = sum(bucket_data['total_days'] for bucket_data in pattern_stats.values())
            retraced = sum(bucket_data['retraced_days'] for bucket_data in pattern_stats.values())
            
            if total > 0:
                rate = (retraced / total) * 100
                pattern_data.append({
                    'Level': level_name,
                    'Total': total,
                    'Retraced': retraced,
                    'Rate_%': round(rate, 2)
                })
    
    if pattern_data:
        pattern_df = pd.DataFrame(pattern_data)
        print(pattern_df.to_string(index=False))

# 4. TOP PATTERN × DISTANCE COMBINATIONS (across all levels)
print("\n\n4. TOP PATTERN × DISTANCE COMBINATIONS (20+ samples, sorted by rate)")
print("-" * 120)

all_combos = []
for level_name in stats.keys():
    for pattern in patterns:
        for bucket_min, bucket_max in DISTANCE_BUCKETS:
            bucket_name = f"{bucket_min:.2f}-{bucket_max:.2f}%" if bucket_max != float('inf') else f"{bucket_min:.2f}%+"
            combo_key = f"{pattern}_{bucket_name}"
            
            if combo_key in stats[level_name]:
                combo_stats = stats[level_name][combo_key]
                total = sum(bucket_data['total_days'] for bucket_data in combo_stats.values())
                retraced = sum(bucket_data['retraced_days'] for bucket_data in combo_stats.values())
                
                if total >= 20:
                    rate = (retraced / total) * 100
                    all_combos.append({
                        'Level': level_name,
                        'Pattern': pattern_labels[pattern],
                        'Distance': bucket_name,
                        'Total': total,
                        'Retraced': retraced,
                        'Rate_%': round(rate, 2)
                    })

if all_combos:
    combos_df = pd.DataFrame(all_combos)
    combos_df = combos_df.sort_values('Rate_%', ascending=False)
    print(combos_df.head(30).to_string(index=False))  # Top 30 combinations

# 5. TIMING ANALYSIS - WHEN DO RETRACEMENTS OCCUR?
print("\n\n5. TIMING ANALYSIS - When during 8am-4pm do retracements occur?")
print("-" * 120)
print("Showing top 5 levels by retracement rate")

for i in range(min(5, len(level_summary_sorted))):
    level_name = level_summary_sorted[i]['Level']
    
    # Aggregate timing data
    overall_timing = defaultdict(int)
    total_retracements = 0
    
    for bucket_data in stats[level_name]['overall'].values():
        for time_slot, count in bucket_data['time_slots'].items():
            overall_timing[time_slot] += count
            total_retracements += count
    
    if total_retracements > 0:
        print(f"\n{level_name} (Total retracements: {total_retracements})")
        
        # Group into hourly buckets for concise display
        hourly_timing = defaultdict(int)
        for time_slot, count in overall_timing.items():
            hour = time_slot.split(':')[0]
            hourly_timing[hour] += count
        
        timing_rows = []
        for hour in sorted(hourly_timing.keys()):
            count = hourly_timing[hour]
            pct = (count / total_retracements) * 100
            timing_rows.append({
                'Hour': f"{hour}:00",
                'Count': count,
                'Pct_%': round(pct, 2)
            })
        
        timing_df = pd.DataFrame(timing_rows)
        print(timing_df.to_string(index=False))

print("\n\n" + "=" * 120)
print("ANALYSIS COMPLETE")
print("=" * 120)
print(f"\nFull dataset exported to: {output_path}")
print("Use the CSV for deeper analysis, filtering, and visualization")

Processing hourly level retracement data...
Tracking 27 levels per day
Distance buckets: 9

Processed 500 days...
Processed 1000 days...
Processed 1500 days...
Processed 2000 days...
Processed 2500 days...
Processed 3000 days...
Processed 3500 days...

Processed 3785 days total

ANALYSIS COMPLETE - Generating Reports

Detailed data exported to: C:\Users\tammm\NQ and ES analysis\hourly_level_retracements.csv



TOP PERFORMING LEVELS - OVERALL

 Rank            Level  Total_Days  Retraced  Rate_%
    1     07:00am_open        3127      2766   88.46
    2 06:00am_midpoint        3130      2731   87.25
    3     06:00am_open        3125      2638   84.42
    4 05:00am_midpoint        3130      2640   84.35
    5     05:00am_open        3124      2584   82.71
    6 04:00am_midpoint        3130      2558   81.73
    7     04:00am_open        3129      2490   79.58
    8 03:00am_midpoint        3130      2468   78.85
    9     03:00am_open        3128      2413   77.14
   10 02:00am_midpoint 

In [5]:
## Hourly Level Retracement Analysis - NY Futures Session (8am-4pm EST)
## Tracks: hourly opens & midpoints from 6pm-7am, distance buckets, timing, London midpoint context

# Define hourly levels to track (6pm previous day through 7am current day)
# Format: (hour, level_type) where hour is 0-23, level_type is 'open' or 'midpoint'
HOURLY_LEVELS = []

# Previous day: 18:00 (6pm) through 23:00 (11pm)
for hour in range(18, 24):
    HOURLY_LEVELS.append((hour, 'open', 'prev_day'))
    HOURLY_LEVELS.append((hour, 'midpoint', 'prev_day'))

# Current day: 00:00 (midnight) through 07:00 (7am)
for hour in range(0, 8):
    HOURLY_LEVELS.append((hour, 'open', 'current_day'))
    if hour < 7:  # 7-8am midpoint is included, but not 8am open
        HOURLY_LEVELS.append((hour, 'midpoint', 'current_day'))

# Distance buckets (in % of price)
DISTANCE_BUCKETS = [
    (0, 0.25),
    (0.25, 0.5),
    (0.5, 0.75),
    (0.75, 1.0),
    (1.0, 1.5),
    (1.5, 2.0),
    (2.0, 3.0),
    (3.0, 5.0),
    (5.0, float('inf'))
]

def get_distance_bucket(distance_pct):
    """Return which bucket a distance percentage falls into"""
    for bucket_min, bucket_max in DISTANCE_BUCKETS:
        if bucket_min <= distance_pct < bucket_max:
            return f"{bucket_min:.2f}-{bucket_max:.2f}%" if bucket_max != float('inf') else f"{bucket_min:.2f}%+"
    return "Unknown"

# Initialize tracking structure
# stats[level_name][context][distance_bucket] = {
#     'total_days': count,
#     'retraced_days': count,
#     'time_slots': {time_slot: count},
#     'adverse_excursion_buckets': {bucket: count}  # NEW: track adverse excursion
# }
stats = defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: {
    'total_days': 0,
    'retraced_days': 0,
    'time_slots': defaultdict(int),
    'adverse_excursion_buckets': defaultdict(int)
})))

print("Processing hourly level retracement data...")
print(f"Tracking {len(HOURLY_LEVELS)} levels per day")
print(f"Distance buckets: {len(DISTANCE_BUCKETS)}")
print()

processed_count = 0

for date in dates[1:]:  # Start from second day
    processed_count += 1
    if processed_count % 500 == 0:
        print(f"Processed {processed_count} days...")
    
    # Map the NY trading day back to the actual date
    actual_date = date + pd.Timedelta(hours=7)
    
    # Define session time ranges
    asian_start = actual_date.replace(hour=20, minute=0, second=0) - pd.Timedelta(days=1)
    asian_end = actual_date.replace(hour=2, minute=0, second=0)
    london_start = asian_end
    london_end = actual_date.replace(hour=8, minute=0, second=0)
    ny_futures_start = london_end  # 8am
    ny_futures_end = actual_date.replace(hour=16, minute=0, second=0)  # 4pm
    
    # Extract session data
    asian_data = df.loc[asian_start:asian_end]
    london_data = df.loc[london_start:london_end]
    ny_futures_data = df.loc[ny_futures_start:ny_futures_end]
    
    # Skip if any session data is missing
    if asian_data.empty or london_data.empty or ny_futures_data.empty:
        continue
    
    # Get 8am open price (for distance calculations)
    open_8am_data = df.loc[ny_futures_start:ny_futures_start + pd.Timedelta(minutes=1)]
    if open_8am_data.empty:
        continue
    open_8am = open_8am_data['Open'].iloc[0]
    
    # Calculate London midpoint (2am-8am range)
    london_high = london_data['High'].max()
    london_low = london_data['Low'].min()
    if pd.isna(london_high) or pd.isna(london_low):
        continue
    london_midpoint = (london_high + london_low) / 2
    
    # Determine pattern (for additional filtering)
    asian_high = asian_data['High'].max()
    asian_low = asian_data['Low'].min()
    
    if pd.isna([asian_high, asian_low]).any():
        continue
    
    current_pattern = 'none'
    if london_high > asian_high and london_low < asian_low:
        current_pattern = 'london_engulfs_asia'
    elif london_high <= asian_high and london_low >= asian_low:
        current_pattern = 'asia_engulfs_london'
    elif london_high > asian_high and london_low >= asian_low:
        current_pattern = 'london_partial_up'
    elif london_low < asian_low and london_high <= asian_high:
        current_pattern = 'london_partial_down'
    
    # Now process each hourly level
    for hour, level_type, day_type in HOURLY_LEVELS:
        # Determine the timestamp for this level
        if day_type == 'prev_day':
            level_time = actual_date.replace(hour=hour, minute=0, second=0) - pd.Timedelta(days=1)
        else:
            level_time = actual_date.replace(hour=hour, minute=0, second=0)
        
        # Calculate the level price
        if level_type == 'open':
            # Get the open at this hour
            level_data = df.loc[level_time:level_time + pd.Timedelta(minutes=1)]
            if level_data.empty:
                continue
            level_price = level_data['Open'].iloc[0]
        else:  # midpoint
            # Get the hourly range and calculate midpoint
            hour_end = level_time + pd.Timedelta(hours=1)
            hour_data = df.loc[level_time:hour_end]
            if hour_data.empty:
                continue
            hour_high = hour_data['High'].max()
            hour_low = hour_data['Low'].min()
            if pd.isna(hour_high) or pd.isna(hour_low):
                continue
            level_price = (hour_high + hour_low) / 2
        
        # Calculate distance from 8am open to this level (as % of price)
        distance_points = abs(open_8am - level_price)
        distance_pct = (distance_points / open_8am) * 100
        distance_bucket = get_distance_bucket(distance_pct)
        
        # Create level name
        hour_12 = hour % 12
        if hour_12 == 0:
            hour_12 = 12
        am_pm = 'pm' if hour >= 12 else 'am'
        level_name = f"{hour_12:02d}:00{am_pm}_{level_type}"
        
        # Check if this level was retraced during NY futures session (8am-4pm)
        # Also track adverse excursion - how far price moved against the level before retracing
        was_retraced = False
        retracement_time = None
        adverse_excursion_pct = 0.0
        
        # Determine direction: is level above or below 8am open?
        level_above_open = level_price > open_8am
        
        # Track the worst adverse move before retracement
        max_adverse_move = 0.0
        
        for idx, row in ny_futures_data.iterrows():
            # Check if we hit the level
            if row['Low'] <= level_price <= row['High']:
                was_retraced = True
                retracement_time = idx
                break
            
            # Track adverse excursion (movement away from level, towards opposite direction)
            if level_above_open:
                # Level is above open, so adverse move is downward (away from level)
                # Most adverse would be the lowest low seen before retracement
                adverse_move_points = level_price - row['Low']
                if adverse_move_points > max_adverse_move:
                    max_adverse_move = adverse_move_points
            else:
                # Level is below open, so adverse move is upward (away from level)
                # Most adverse would be the highest high seen before retracement
                adverse_move_points = row['High'] - level_price
                if adverse_move_points > max_adverse_move:
                    max_adverse_move = adverse_move_points
        
        # Convert adverse excursion to percentage
        if was_retraced and max_adverse_move > 0:
            adverse_excursion_pct = (max_adverse_move / open_8am) * 100
        
        # Update statistics for different contexts
        contexts_to_update = [
            'overall',
            current_pattern,
            f"{current_pattern}_{distance_bucket}"  # Pattern + distance combination
        ]
        
        for context in contexts_to_update:
            stats[level_name][context][distance_bucket]['total_days'] += 1
            
            if was_retraced:
                stats[level_name][context][distance_bucket]['retraced_days'] += 1
                
                # Track timing (15-minute slots)
                hour = retracement_time.hour
                minute = retracement_time.minute
                slot_minute = (minute // 15) * 15
                time_slot = f"{hour:02d}:{slot_minute:02d}"
                stats[level_name][context][distance_bucket]['time_slots'][time_slot] += 1
                
                # Track adverse excursion bucket
                adverse_bucket = get_distance_bucket(adverse_excursion_pct)
                stats[level_name][context][distance_bucket]['adverse_excursion_buckets'][adverse_bucket] += 1

print(f"\nProcessed {processed_count} days total")
print("\n" + "=" * 120)
print("ANALYSIS COMPLETE - Generating Reports")
print("=" * 120)

# ========================================================================
# DISPLAY FUNCTIONS
# ========================================================================

def display_level_summary(level_name, level_stats):
    """Display summary statistics for a single level across all contexts"""
    print("\n" + "=" * 120)
    print(f"LEVEL: {level_name}")
    print("=" * 120)
    
    # Overall stats first
    overall_stats = level_stats['overall']
    
    print("\nOVERALL STATISTICS (all days combined):")
    print("-" * 120)
    
    total_overall = sum(bucket_data['total_days'] for bucket_data in overall_stats.values())
    retraced_overall = sum(bucket_data['retraced_days'] for bucket_data in overall_stats.values())
    
    if total_overall > 0:
        overall_pct = (retraced_overall / total_overall) * 100
        print(f"Total days: {total_overall} | Retraced: {retraced_overall} | Rate: {overall_pct:.2f}%")
        
        print("\nBy Distance Bucket:")
        
        # Build table data
        bucket_rows = []
        for bucket_min, bucket_max in DISTANCE_BUCKETS:
            bucket_name = f"{bucket_min:.2f}-{bucket_max:.2f}%" if bucket_max != float('inf') else f"{bucket_min:.2f}%+"
            
            if bucket_name in overall_stats:
                data = overall_stats[bucket_name]
                if data['total_days'] > 0:
                    rate = (data['retraced_days'] / data['total_days']) * 100
                    bucket_rows.append({
                        'Distance_Bucket': bucket_name,
                        'Total': data['total_days'],
                        'Retraced': data['retraced_days'],
                        'Rate_%': round(rate, 2)
                    })
        
        if bucket_rows:
            bucket_df = pd.DataFrame(bucket_rows)
            print(bucket_df.to_string(index=False))
    
    # Pattern context stats
    print("\n\nBY SESSION PATTERN:")
    print("-" * 120)
    
    patterns = ['london_engulfs_asia', 'asia_engulfs_london', 'london_partial_up', 'london_partial_down']
    pattern_labels = {
        'london_engulfs_asia': 'London Engulfs Asia',
        'asia_engulfs_london': 'Asia Engulfs London',
        'london_partial_up': 'London Partial Engulf Up',
        'london_partial_down': 'London Partial Engulf Down'
    }
    
    pattern_rows = []
    for pattern in patterns:
        pattern_stats = level_stats[pattern]
        total = sum(bucket_data['total_days'] for bucket_data in pattern_stats.values())
        retraced = sum(bucket_data['retraced_days'] for bucket_data in pattern_stats.values())
        
        if total > 0:
            rate = (retraced / total) * 100
            pattern_rows.append({
                'Pattern': pattern_labels[pattern],
                'Total': total,
                'Retraced': retraced,
                'Rate_%': round(rate, 2)
            })
    
    if pattern_rows:
        pattern_df = pd.DataFrame(pattern_rows)
        print(pattern_df.to_string(index=False))
    
    # Pattern × Distance combinations
    print("\n\nPATTERN × DISTANCE COMBINATIONS (showing combinations with 20+ samples):")
    print("-" * 120)
    
    combo_rows = []
    for pattern in patterns:
        for bucket_min, bucket_max in DISTANCE_BUCKETS:
            bucket_name = f"{bucket_min:.2f}-{bucket_max:.2f}%" if bucket_max != float('inf') else f"{bucket_min:.2f}%+"
            combo_key = f"{pattern}_{bucket_name}"
            
            if combo_key in level_stats:
                combo_stats = level_stats[combo_key]
                total = sum(bucket_data['total_days'] for bucket_data in combo_stats.values())
                retraced = sum(bucket_data['retraced_days'] for bucket_data in combo_stats.values())
                
                if total >= 20:  # Only show combinations with sufficient sample size
                    rate = (retraced / total) * 100
                    combo_rows.append({
                        'Pattern': pattern_labels[pattern],
                        'Distance': bucket_name,
                        'Total': total,
                        'Retraced': retraced,
                        'Rate_%': round(rate, 2)
                    })
    
    if combo_rows:
        # Sort by rate descending
        combo_df = pd.DataFrame(combo_rows)
        combo_df = combo_df.sort_values('Rate_%', ascending=False)
        print(combo_df.to_string(index=False))
    else:
        print("No pattern-distance combinations with 20+ samples found.")


def display_timing_analysis(level_name, level_stats):
    """Display timing analysis for when retracements occur"""
    print("\n" + "=" * 120)
    print(f"TIMING ANALYSIS: {level_name}")
    print("When during 8am-4pm session does retracement occur?")
    print("=" * 120)
    
    # Aggregate all timing data across all buckets for overall context
    overall_timing = defaultdict(int)
    total_retracements = 0
    
    for bucket_data in level_stats['overall'].values():
        for time_slot, count in bucket_data['time_slots'].items():
            overall_timing[time_slot] += count
            total_retracements += count
    
    if total_retracements > 0:
        print(f"\nTotal retracements tracked: {total_retracements}")
        
        # Build timing table
        timing_rows = []
        sorted_slots = sorted(overall_timing.items())
        for time_slot, count in sorted_slots:
            pct = (count / total_retracements) * 100
            timing_rows.append({
                'Time_Slot': time_slot,
                'Count': count,
                'Pct_of_Total': round(pct, 2)
            })
        
        timing_df = pd.DataFrame(timing_rows)
        print("\n" + timing_df.to_string(index=False))


# ========================================================================
# EXPORT TO CSV
# ========================================================================

export_data = []

for level_name, level_stats in stats.items():
    for context, context_stats in level_stats.items():
        for distance_bucket, bucket_data in context_stats.items():
            if bucket_data['total_days'] > 0:
                rate = (bucket_data['retraced_days'] / bucket_data['total_days']) * 100
                
                export_data.append({
                    'Level': level_name,
                    'Context': context,
                    'Distance_Bucket': distance_bucket,
                    'Total_Days': bucket_data['total_days'],
                    'Retraced_Days': bucket_data['retraced_days'],
                    'Retracement_Rate_Pct': rate
                })

export_df = pd.DataFrame(export_data)
output_path = r"C:\Users\tammm\NQ and ES analysis\hourly_level_retracements.csv"
export_df.to_csv(output_path, index=False)
print(f"\nDetailed data exported to: {output_path}")

# ========================================================================
# GENERATE SUMMARY REPORT
# ========================================================================

print("\n\n")
print("=" * 120)
print("TOP PERFORMING LEVELS - OVERALL")
print("=" * 120)

# Calculate overall stats for each level and sort by retracement rate
level_summary = []
for level_name, level_stats in stats.items():
    overall_stats = level_stats['overall']
    total = sum(bucket_data['total_days'] for bucket_data in overall_stats.values())
    retraced = sum(bucket_data['retraced_days'] for bucket_data in overall_stats.values())
    
    if total > 0:
        rate = (retraced / total) * 100
        level_summary.append({
            'Level': level_name,
            'Total_Days': total,
            'Retraced': retraced,
            'Rate_%': round(rate, 2)
        })

# Sort by retracement rate
level_summary_sorted = sorted(level_summary, key=lambda x: x['Rate_%'], reverse=True)

# Create DataFrame for top 20
top_20_df = pd.DataFrame(level_summary_sorted[:20])
top_20_df.insert(0, 'Rank', range(1, len(top_20_df) + 1))
print("\n" + top_20_df.to_string(index=False))

# ========================================================================
# COMPREHENSIVE ANALYSIS - ALL LEVELS IN TABLE FORMAT
# ========================================================================

print("\n\n")
print("=" * 120)
print("COMPREHENSIVE ANALYSIS - ALL LEVELS")
print("=" * 120)

# Sort levels by time order for easier reading
level_time_order = []
for hour in range(18, 24):  # 6pm-11pm
    level_time_order.append(f"{hour % 12 if hour % 12 != 0 else 12:02d}:00pm_open")
    level_time_order.append(f"{hour % 12 if hour % 12 != 0 else 12:02d}:00pm_midpoint")
for hour in range(0, 8):  # 12am-7am
    am_hour = hour if hour != 0 else 12
    level_time_order.append(f"{am_hour:02d}:00am_open")
    if hour < 7:
        level_time_order.append(f"{am_hour:02d}:00am_midpoint")

# 1. OVERALL RETRACEMENT RATES
print("\n1. OVERALL RETRACEMENT RATES")
print("-" * 120)

overall_data = []
for level_name in level_time_order:
    if level_name in stats:
        overall_stats = stats[level_name]['overall']
        total = sum(bucket_data['total_days'] for bucket_data in overall_stats.values())
        retraced = sum(bucket_data['retraced_days'] for bucket_data in overall_stats.values())
        
        if total > 0:
            rate = (retraced / total) * 100
            overall_data.append({
                'Level': level_name,
                'Total': total,
                'Retraced': retraced,
                'Rate_%': round(rate, 2)
            })

overall_df = pd.DataFrame(overall_data)
print(overall_df.to_string(index=False))

# 2. RETRACEMENT RATES BY DISTANCE BUCKET
print("\n\n2. RETRACEMENT RATES BY DISTANCE BUCKET")
print("-" * 120)

for bucket_min, bucket_max in DISTANCE_BUCKETS[:5]:  # Show first 5 buckets (most common)
    bucket_name = f"{bucket_min:.2f}-{bucket_max:.2f}%" if bucket_max != float('inf') else f"{bucket_min:.2f}%+"
    
    print(f"\nDistance: {bucket_name}")
    bucket_data = []
    
    for level_name in level_time_order:
        if level_name in stats and bucket_name in stats[level_name]['overall']:
            data = stats[level_name]['overall'][bucket_name]
            if data['total_days'] > 0:
                rate = (data['retraced_days'] / data['total_days']) * 100
                bucket_data.append({
                    'Level': level_name,
                    'Total': data['total_days'],
                    'Retraced': data['retraced_days'],
                    'Rate_%': round(rate, 2)
                })
    
    if bucket_data:
        bucket_df = pd.DataFrame(bucket_data)
        print(bucket_df.to_string(index=False))

# 3. RETRACEMENT RATES BY SESSION PATTERN
print("\n\n3. RETRACEMENT RATES BY SESSION PATTERN")
print("-" * 120)

patterns = ['london_engulfs_asia', 'asia_engulfs_london', 'london_partial_up', 'london_partial_down']
pattern_labels = {
    'london_engulfs_asia': 'London Engulfs Asia',
    'asia_engulfs_london': 'Asia Engulfs London',
    'london_partial_up': 'London Partial Up',
    'london_partial_down': 'London Partial Down'
}

for pattern in patterns:
    print(f"\nPattern: {pattern_labels[pattern]}")
    pattern_data = []
    
    for level_name in level_time_order:
        if level_name in stats and pattern in stats[level_name]:
            pattern_stats = stats[level_name][pattern]
            total = sum(bucket_data['total_days'] for bucket_data in pattern_stats.values())
            retraced = sum(bucket_data['retraced_days'] for bucket_data in pattern_stats.values())
            
            if total > 0:
                rate = (retraced / total) * 100
                pattern_data.append({
                    'Level': level_name,
                    'Total': total,
                    'Retraced': retraced,
                    'Rate_%': round(rate, 2)
                })
    
    if pattern_data:
        pattern_df = pd.DataFrame(pattern_data)
        print(pattern_df.to_string(index=False))

# 4. TOP PATTERN × DISTANCE COMBINATIONS (across all levels)
print("\n\n4. TOP PATTERN × DISTANCE COMBINATIONS (20+ samples, sorted by rate)")
print("-" * 120)

all_combos = []
for level_name in stats.keys():
    for pattern in patterns:
        for bucket_min, bucket_max in DISTANCE_BUCKETS:
            bucket_name = f"{bucket_min:.2f}-{bucket_max:.2f}%" if bucket_max != float('inf') else f"{bucket_min:.2f}%+"
            combo_key = f"{pattern}_{bucket_name}"
            
            if combo_key in stats[level_name]:
                combo_stats = stats[level_name][combo_key]
                total = sum(bucket_data['total_days'] for bucket_data in combo_stats.values())
                retraced = sum(bucket_data['retraced_days'] for bucket_data in combo_stats.values())
                
                if total >= 20:
                    rate = (retraced / total) * 100
                    all_combos.append({
                        'Level': level_name,
                        'Pattern': pattern_labels[pattern],
                        'Distance': bucket_name,
                        'Total': total,
                        'Retraced': retraced,
                        'Rate_%': round(rate, 2)
                    })

if all_combos:
    combos_df = pd.DataFrame(all_combos)
    combos_df = combos_df.sort_values('Rate_%', ascending=False)
    print(combos_df.head(30).to_string(index=False))  # Top 30 combinations

# 5. TIMING ANALYSIS - WHEN DO RETRACEMENTS OCCUR?
print("\n\n5. TIMING ANALYSIS - When during 8am-4pm do retracements occur?")
print("-" * 120)
print("Showing top 5 levels by retracement rate")

for i in range(min(5, len(level_summary_sorted))):
    level_name = level_summary_sorted[i]['Level']
    
    # Aggregate timing data
    overall_timing = defaultdict(int)
    total_retracements = 0
    
    for bucket_data in stats[level_name]['overall'].values():
        for time_slot, count in bucket_data['time_slots'].items():
            overall_timing[time_slot] += count
            total_retracements += count
    
    if total_retracements > 0:
        print(f"\n{level_name} (Total retracements: {total_retracements})")
        
        # Group into hourly buckets for concise display
        hourly_timing = defaultdict(int)
        for time_slot, count in overall_timing.items():
            hour = time_slot.split(':')[0]
            hourly_timing[hour] += count
        
        timing_rows = []
        for hour in sorted(hourly_timing.keys()):
            count = hourly_timing[hour]
            pct = (count / total_retracements) * 100
            timing_rows.append({
                'Hour': f"{hour}:00",
                'Count': count,
                'Pct_%': round(pct, 2)
            })
        
        timing_df = pd.DataFrame(timing_rows)
        print(timing_df.to_string(index=False))

# 6. ADVERSE EXCURSION ANALYSIS
print("\n\n6. ADVERSE EXCURSION ANALYSIS - How far does price move against the level before retracing?")
print("-" * 120)
print("For retracements that occurred, what was the maximum adverse move (in % terms)?")
print("This shows how much 'heat' you need to tolerate before the retracement happens.\n")

# Analyze adverse excursion for top levels
for i in range(min(10, len(level_summary_sorted))):
    level_name = level_summary_sorted[i]['Level']
    
    # Aggregate adverse excursion data across all distance buckets
    adverse_excursion_totals = defaultdict(int)
    total_retracements = 0
    
    for bucket_data in stats[level_name]['overall'].values():
        for adverse_bucket, count in bucket_data['adverse_excursion_buckets'].items():
            adverse_excursion_totals[adverse_bucket] += count
            total_retracements += count
    
    if total_retracements > 0:
        print(f"\n{level_name} (Total retracements: {total_retracements})")
        
        adverse_rows = []
        for adverse_bucket in sorted(adverse_excursion_totals.keys()):
            count = adverse_excursion_totals[adverse_bucket]
            pct = (count / total_retracements) * 100
            adverse_rows.append({
                'Adverse_Move': adverse_bucket,
                'Count': count,
                'Pct_%': round(pct, 2)
            })
        
        if adverse_rows:
            adverse_df = pd.DataFrame(adverse_rows)
            print(adverse_df.to_string(index=False))

# 7. ADVERSE EXCURSION BY DISTANCE BUCKET
print("\n\n7. ADVERSE EXCURSION BY DISTANCE FROM LEVEL")
print("-" * 120)
print("How does adverse excursion vary based on how far away the level is?")
print("Showing data for top 3 levels\n")

for i in range(min(3, len(level_summary_sorted))):
    level_name = level_summary_sorted[i]['Level']
    
    print(f"\n{level_name}:")
    print("-" * 120)
    
    # For each distance bucket, show adverse excursion distribution
    for bucket_min, bucket_max in DISTANCE_BUCKETS[:4]:  # First 4 distance buckets
        bucket_name = f"{bucket_min:.2f}-{bucket_max:.2f}%" if bucket_max != float('inf') else f"{bucket_min:.2f}%+"
        
        if bucket_name in stats[level_name]['overall']:
            bucket_data = stats[level_name]['overall'][bucket_name]
            
            if bucket_data['retraced_days'] > 0:
                print(f"\nDistance from level: {bucket_name} (Retracements: {bucket_data['retraced_days']})")
                
                adverse_rows = []
                for adverse_bucket, count in bucket_data['adverse_excursion_buckets'].items():
                    pct = (count / bucket_data['retraced_days']) * 100
                    adverse_rows.append({
                        'Adverse_Move': adverse_bucket,
                        'Count': count,
                        'Pct_%': round(pct, 2)
                    })
                
                if adverse_rows:
                    adverse_df = pd.DataFrame(adverse_rows)
                    print(adverse_df.to_string(index=False))

print("\n\n" + "=" * 120)
print("ANALYSIS COMPLETE")
print("=" * 120)
print(f"\nFull dataset exported to: {output_path}")
print("Use the CSV for deeper analysis, filtering, and visualization")

Processing hourly level retracement data...
Tracking 27 levels per day
Distance buckets: 9

Processed 500 days...
Processed 1000 days...
Processed 1500 days...
Processed 2000 days...
Processed 2500 days...
Processed 3000 days...
Processed 3500 days...

Processed 3785 days total

ANALYSIS COMPLETE - Generating Reports

Detailed data exported to: C:\Users\tammm\NQ and ES analysis\hourly_level_retracements.csv



TOP PERFORMING LEVELS - OVERALL

 Rank            Level  Total_Days  Retraced  Rate_%
    1     07:00am_open        3127      2766   88.46
    2 06:00am_midpoint        3130      2731   87.25
    3     06:00am_open        3125      2638   84.42
    4 05:00am_midpoint        3130      2640   84.35
    5     05:00am_open        3124      2584   82.71
    6 04:00am_midpoint        3130      2558   81.73
    7     04:00am_open        3129      2490   79.58
    8 03:00am_midpoint        3130      2468   78.85
    9     03:00am_open        3128      2413   77.14
   10 02:00am_midpoint 

Adverse_Move  Count  Pct_%
  0.00-0.25%   1114  46.17
  0.25-0.50%    627  25.98
  0.50-0.75%    288  11.94
  0.75-1.00%    169   7.00
  1.00-1.50%    133   5.51
  1.50-2.00%     45   1.86
  2.00-3.00%     27   1.12
  3.00-5.00%      8   0.33
      5.00%+      2   0.08

02:00am_midpoint (Total retracements: 2400)
Adverse_Move  Count  Pct_%
  0.00-0.25%   1090  45.42
  0.25-0.50%    650  27.08
  0.50-0.75%    288  12.00
  0.75-1.00%    167   6.96
  1.00-1.50%    131   5.46
  1.50-2.00%     43   1.79
  2.00-3.00%     20   0.83
  3.00-5.00%     10   0.42
      5.00%+      1   0.04


7. ADVERSE EXCURSION BY DISTANCE FROM LEVEL
------------------------------------------------------------------------------------------------------------------------
How does adverse excursion vary based on how far away the level is?
Showing data for top 3 levels


07:00am_open:
------------------------------------------------------------------------------------------------------------------------

Distance fro

In [6]:
## Hourly Level Retracement Analysis - NY Futures Session (8am-4pm EST)
## Tracks: hourly opens & midpoints from 6pm-7am, distance buckets, timing, London midpoint context

# Define hourly levels to track (6pm previous day through 7am current day)
# Format: (hour, level_type) where hour is 0-23, level_type is 'open' or 'midpoint'
HOURLY_LEVELS = []

# Previous day: 18:00 (6pm) through 23:00 (11pm)
for hour in range(18, 24):
    HOURLY_LEVELS.append((hour, 'open', 'prev_day'))
    HOURLY_LEVELS.append((hour, 'midpoint', 'prev_day'))

# Current day: 00:00 (midnight) through 07:00 (7am)
for hour in range(0, 8):
    HOURLY_LEVELS.append((hour, 'open', 'current_day'))
    if hour <= 7:  # Include midpoints for all hours 0-7 (including 7am midpoint)
        HOURLY_LEVELS.append((hour, 'midpoint', 'current_day'))

# Distance buckets (in % of price)
DISTANCE_BUCKETS = [
    (0, 0.25),
    (0.25, 0.5),
    (0.5, 0.75),
    (0.75, 1.0),
    (1.0, 1.5),
    (1.5, 2.0),
    (2.0, 3.0),
    (3.0, 5.0),
    (5.0, float('inf'))
]

def get_distance_bucket(distance_pct):
    """Return which bucket a distance percentage falls into"""
    for bucket_min, bucket_max in DISTANCE_BUCKETS:
        if bucket_min <= distance_pct < bucket_max:
            return f"{bucket_min:.2f}-{bucket_max:.2f}%" if bucket_max != float('inf') else f"{bucket_min:.2f}%+"
    return "Unknown"

# Initialize tracking structure
# stats[level_name][context][distance_bucket] = {
#     'total_days': count,
#     'retraced_days': count,
#     'time_slots': {time_slot: count},
#     'adverse_excursion_buckets': {bucket: count}
# }
stats = defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: {
    'total_days': 0,
    'retraced_days': 0,
    'time_slots': defaultdict(int),
    'adverse_excursion_buckets': defaultdict(int)
})))

print("Processing hourly level retracement data...")
print(f"Tracking {len(HOURLY_LEVELS)} levels per day")
print(f"Distance buckets: {len(DISTANCE_BUCKETS)}")
print()

processed_count = 0

for date in dates[1:]:  # Start from second day
    processed_count += 1
    if processed_count % 500 == 0:
        print(f"Processed {processed_count} days...")
    
    # Map the NY trading day back to the actual date
    actual_date = date + pd.Timedelta(hours=7)
    
    # Define session time ranges
    asian_start = actual_date.replace(hour=20, minute=0, second=0) - pd.Timedelta(days=1)
    asian_end = actual_date.replace(hour=2, minute=0, second=0)
    london_start = asian_end
    london_end = actual_date.replace(hour=8, minute=0, second=0)
    ny_futures_start = london_end  # 8am
    ny_futures_end = actual_date.replace(hour=16, minute=0, second=0)  # 4pm
    
    # Extract session data
    asian_data = df.loc[asian_start:asian_end]
    london_data = df.loc[london_start:london_end]
    ny_futures_data = df.loc[ny_futures_start:ny_futures_end]
    
    # Skip if any session data is missing
    if asian_data.empty or london_data.empty or ny_futures_data.empty:
        continue
    
    # Get 8am open price (for distance calculations)
    open_8am_data = df.loc[ny_futures_start:ny_futures_start + pd.Timedelta(minutes=1)]
    if open_8am_data.empty:
        continue
    open_8am = open_8am_data['Open'].iloc[0]
    
    # Calculate London midpoint (2am-8am range)
    london_high = london_data['High'].max()
    london_low = london_data['Low'].min()
    if pd.isna(london_high) or pd.isna(london_low):
        continue
    london_midpoint = (london_high + london_low) / 2
    
    # Determine pattern (for additional filtering)
    asian_high = asian_data['High'].max()
    asian_low = asian_data['Low'].min()
    
    if pd.isna([asian_high, asian_low]).any():
        continue
    
    current_pattern = 'none'
    if london_high > asian_high and london_low < asian_low:
        current_pattern = 'london_engulfs_asia'
    elif london_high <= asian_high and london_low >= asian_low:
        current_pattern = 'asia_engulfs_london'
    elif london_high > asian_high and london_low >= asian_low:
        current_pattern = 'london_partial_up'
    elif london_low < asian_low and london_high <= asian_high:
        current_pattern = 'london_partial_down'
    
    # Now process each hourly level
    for hour, level_type, day_type in HOURLY_LEVELS:
        # Determine the timestamp for this level
        if day_type == 'prev_day':
            level_time = actual_date.replace(hour=hour, minute=0, second=0) - pd.Timedelta(days=1)
        else:
            level_time = actual_date.replace(hour=hour, minute=0, second=0)
        
        # Calculate the level price
        if level_type == 'open':
            # Get the open at this hour
            level_data = df.loc[level_time:level_time + pd.Timedelta(minutes=1)]
            if level_data.empty:
                continue
            level_price = level_data['Open'].iloc[0]
        else:  # midpoint
            # Get the hourly range and calculate midpoint
            hour_end = level_time + pd.Timedelta(hours=1)
            hour_data = df.loc[level_time:hour_end]
            if hour_data.empty:
                continue
            hour_high = hour_data['High'].max()
            hour_low = hour_data['Low'].min()
            if pd.isna(hour_high) or pd.isna(hour_low):
                continue
            level_price = (hour_high + hour_low) / 2
        
        # Calculate distance from 8am open to this level (as % of price)
        distance_points = abs(open_8am - level_price)
        distance_pct = (distance_points / open_8am) * 100
        distance_bucket = get_distance_bucket(distance_pct)
        
        # Create level name
        hour_12 = hour % 12
        if hour_12 == 0:
            hour_12 = 12
        am_pm = 'pm' if hour >= 12 else 'am'
        level_name = f"{hour_12:02d}:00{am_pm}_{level_type}"
        
        # Check if this level was retraced during NY futures session (8am-4pm)
        # Also track adverse excursion - how far price moved against the level before retracing
        was_retraced = False
        retracement_time = None
        adverse_excursion_pct = 0.0
        
        # Determine direction: is level above or below 8am open?
        level_above_open = level_price > open_8am
        
        # Track the worst adverse move before retracement
        max_adverse_move = 0.0
        
        for idx, row in ny_futures_data.iterrows():
            # Check if we hit the level
            if row['Low'] <= level_price <= row['High']:
                was_retraced = True
                retracement_time = idx
                break
            
            # Track adverse excursion (movement away from level, towards opposite direction)
            if level_above_open:
                # Level is above open, so adverse move is downward (away from level)
                # Most adverse would be the lowest low seen before retracement
                adverse_move_points = level_price - row['Low']
                if adverse_move_points > max_adverse_move:
                    max_adverse_move = adverse_move_points
            else:
                # Level is below open, so adverse move is upward (away from level)
                # Most adverse would be the highest high seen before retracement
                adverse_move_points = row['High'] - level_price
                if adverse_move_points > max_adverse_move:
                    max_adverse_move = adverse_move_points
        
        # Convert adverse excursion to percentage
        if was_retraced and max_adverse_move > 0:
            adverse_excursion_pct = (max_adverse_move / open_8am) * 100
        
        # Update statistics for different contexts
        contexts_to_update = [
            'overall',
            current_pattern,
            f"{current_pattern}_{distance_bucket}"  # Pattern + distance combination
        ]
        
        for context in contexts_to_update:
            stats[level_name][context][distance_bucket]['total_days'] += 1
            
            if was_retraced:
                stats[level_name][context][distance_bucket]['retraced_days'] += 1
                
                # Track timing (15-minute slots)
                hour = retracement_time.hour
                minute = retracement_time.minute
                slot_minute = (minute // 15) * 15
                time_slot = f"{hour:02d}:{slot_minute:02d}"
                stats[level_name][context][distance_bucket]['time_slots'][time_slot] += 1
                
                # Track adverse excursion bucket
                adverse_bucket = get_distance_bucket(adverse_excursion_pct)
                stats[level_name][context][distance_bucket]['adverse_excursion_buckets'][adverse_bucket] += 1

print(f"\nProcessed {processed_count} days total")
print("\n" + "=" * 120)
print("ANALYSIS COMPLETE - Generating Reports")
print("=" * 120)

# ========================================================================
# DISPLAY FUNCTIONS
# ========================================================================

def display_level_summary(level_name, level_stats):
    """Display summary statistics for a single level across all contexts"""
    print("\n" + "=" * 120)
    print(f"LEVEL: {level_name}")
    print("=" * 120)
    
    # Overall stats first
    overall_stats = level_stats['overall']
    
    print("\nOVERALL STATISTICS (all days combined):")
    print("-" * 120)
    
    total_overall = sum(bucket_data['total_days'] for bucket_data in overall_stats.values())
    retraced_overall = sum(bucket_data['retraced_days'] for bucket_data in overall_stats.values())
    
    if total_overall > 0:
        overall_pct = (retraced_overall / total_overall) * 100
        print(f"Total days: {total_overall} | Retraced: {retraced_overall} | Rate: {overall_pct:.2f}%")
        
        print("\nBy Distance Bucket:")
        
        # Build table data
        bucket_rows = []
        for bucket_min, bucket_max in DISTANCE_BUCKETS:
            bucket_name = f"{bucket_min:.2f}-{bucket_max:.2f}%" if bucket_max != float('inf') else f"{bucket_min:.2f}%+"
            
            if bucket_name in overall_stats:
                data = overall_stats[bucket_name]
                if data['total_days'] > 0:
                    rate = (data['retraced_days'] / data['total_days']) * 100
                    bucket_rows.append({
                        'Distance_Bucket': bucket_name,
                        'Total': data['total_days'],
                        'Retraced': data['retraced_days'],
                        'Rate_%': round(rate, 2)
                    })
        
        if bucket_rows:
            bucket_df = pd.DataFrame(bucket_rows)
            print(bucket_df.to_string(index=False))
    
    # Pattern context stats
    print("\n\nBY SESSION PATTERN:")
    print("-" * 120)
    
    patterns = ['london_engulfs_asia', 'asia_engulfs_london', 'london_partial_up', 'london_partial_down']
    pattern_labels = {
        'london_engulfs_asia': 'London Engulfs Asia',
        'asia_engulfs_london': 'Asia Engulfs London',
        'london_partial_up': 'London Partial Engulf Up',
        'london_partial_down': 'London Partial Engulf Down'
    }
    
    pattern_rows = []
    for pattern in patterns:
        pattern_stats = level_stats[pattern]
        total = sum(bucket_data['total_days'] for bucket_data in pattern_stats.values())
        retraced = sum(bucket_data['retraced_days'] for bucket_data in pattern_stats.values())
        
        if total > 0:
            rate = (retraced / total) * 100
            pattern_rows.append({
                'Pattern': pattern_labels[pattern],
                'Total': total,
                'Retraced': retraced,
                'Rate_%': round(rate, 2)
            })
    
    if pattern_rows:
        pattern_df = pd.DataFrame(pattern_rows)
        print(pattern_df.to_string(index=False))
    
    # Pattern × Distance combinations
    print("\n\nPATTERN × DISTANCE COMBINATIONS (showing combinations with 20+ samples):")
    print("-" * 120)
    
    combo_rows = []
    for pattern in patterns:
        for bucket_min, bucket_max in DISTANCE_BUCKETS:
            bucket_name = f"{bucket_min:.2f}-{bucket_max:.2f}%" if bucket_max != float('inf') else f"{bucket_min:.2f}%+"
            combo_key = f"{pattern}_{bucket_name}"
            
            if combo_key in level_stats:
                combo_stats = level_stats[combo_key]
                total = sum(bucket_data['total_days'] for bucket_data in combo_stats.values())
                retraced = sum(bucket_data['retraced_days'] for bucket_data in combo_stats.values())
                
                if total >= 20:  # Only show combinations with sufficient sample size
                    rate = (retraced / total) * 100
                    combo_rows.append({
                        'Pattern': pattern_labels[pattern],
                        'Distance': bucket_name,
                        'Total': total,
                        'Retraced': retraced,
                        'Rate_%': round(rate, 2)
                    })
    
    if combo_rows:
        # Sort by rate descending
        combo_df = pd.DataFrame(combo_rows)
        combo_df = combo_df.sort_values('Rate_%', ascending=False)
        print(combo_df.to_string(index=False))
    else:
        print("No pattern-distance combinations with 20+ samples found.")


def display_timing_analysis(level_name, level_stats):
    """Display timing analysis for when retracements occur"""
    print("\n" + "=" * 120)
    print(f"TIMING ANALYSIS: {level_name}")
    print("When during 8am-4pm session does retracement occur?")
    print("=" * 120)
    
    # Aggregate all timing data across all buckets for overall context
    overall_timing = defaultdict(int)
    total_retracements = 0
    
    for bucket_data in level_stats['overall'].values():
        for time_slot, count in bucket_data['time_slots'].items():
            overall_timing[time_slot] += count
            total_retracements += count
    
    if total_retracements > 0:
        print(f"\nTotal retracements tracked: {total_retracements}")
        
        # Build timing table
        timing_rows = []
        sorted_slots = sorted(overall_timing.items())
        for time_slot, count in sorted_slots:
            pct = (count / total_retracements) * 100
            timing_rows.append({
                'Time_Slot': time_slot,
                'Count': count,
                'Pct_of_Total': round(pct, 2)
            })
        
        timing_df = pd.DataFrame(timing_rows)
        print("\n" + timing_df.to_string(index=False))


# ========================================================================
# EXPORT TO CSV
# ========================================================================

export_data = []

for level_name, level_stats in stats.items():
    for context, context_stats in level_stats.items():
        for distance_bucket, bucket_data in context_stats.items():
            if bucket_data['total_days'] > 0:
                rate = (bucket_data['retraced_days'] / bucket_data['total_days']) * 100
                
                export_data.append({
                    'Level': level_name,
                    'Context': context,
                    'Distance_Bucket': distance_bucket,
                    'Total_Days': bucket_data['total_days'],
                    'Retraced_Days': bucket_data['retraced_days'],
                    'Retracement_Rate_Pct': rate
                })

export_df = pd.DataFrame(export_data)
output_path = r"C:\Users\tammm\NQ and ES analysis\hourly_level_retracements.csv"
export_df.to_csv(output_path, index=False)
print(f"\nDetailed data exported to: {output_path}")

# ========================================================================
# GENERATE SUMMARY REPORT
# ========================================================================

print("\n\n")
print("=" * 120)
print("TOP PERFORMING LEVELS - OVERALL")
print("=" * 120)

# Calculate overall stats for each level and sort by retracement rate
level_summary = []
for level_name, level_stats in stats.items():
    overall_stats = level_stats['overall']
    total = sum(bucket_data['total_days'] for bucket_data in overall_stats.values())
    retraced = sum(bucket_data['retraced_days'] for bucket_data in overall_stats.values())
    
    if total > 0:
        rate = (retraced / total) * 100
        level_summary.append({
            'Level': level_name,
            'Total_Days': total,
            'Retraced': retraced,
            'Rate_%': round(rate, 2)
        })

# Sort by retracement rate
level_summary_sorted = sorted(level_summary, key=lambda x: x['Rate_%'], reverse=True)

# Create DataFrame for top 20
top_20_df = pd.DataFrame(level_summary_sorted[:20])
top_20_df.insert(0, 'Rank', range(1, len(top_20_df) + 1))
print("\n" + top_20_df.to_string(index=False))

# ========================================================================
# COMPREHENSIVE ANALYSIS - ALL LEVELS IN TABLE FORMAT
# ========================================================================

print("\n\n")
print("=" * 120)
print("COMPREHENSIVE ANALYSIS - ALL LEVELS")
print("=" * 120)

# Sort levels by time order for easier reading
level_time_order = []
for hour in range(18, 24):  # 6pm-11pm
    level_time_order.append(f"{hour % 12 if hour % 12 != 0 else 12:02d}:00pm_open")
    level_time_order.append(f"{hour % 12 if hour % 12 != 0 else 12:02d}:00pm_midpoint")
for hour in range(0, 8):  # 12am-7am
    am_hour = hour if hour != 0 else 12
    level_time_order.append(f"{am_hour:02d}:00am_open")
    level_time_order.append(f"{am_hour:02d}:00am_midpoint")  # Now includes 7am midpoint

# Helper function to format rate with sample size
def format_rate_sample(retraced, total):
    if total == 0:
        return "-"
    rate = (retraced / total) * 100
    return f"{rate:.1f}% ({total})"

# 1. OVERALL RETRACEMENT RATES - TRANSPOSED TABLE
print("\n1. OVERALL RETRACEMENT RATES")
print("-" * 120)
print("Format: Rate% (Sample Size)")
print()

# Build data for transpose
overall_data = {}
for level_name in level_time_order:
    if level_name in stats:
        overall_stats = stats[level_name]['overall']
        total = sum(bucket_data['total_days'] for bucket_data in overall_stats.values())
        retraced = sum(bucket_data['retraced_days'] for bucket_data in overall_stats.values())
        overall_data[level_name] = format_rate_sample(retraced, total)

# Create DataFrame with levels as columns
overall_df = pd.DataFrame([overall_data])
overall_df.index = ['Overall Rate']
print(overall_df.T.to_string())  # Transpose to show levels as rows (easier to read)

# 2. RETRACEMENT RATES BY DISTANCE BUCKET - TRANSPOSED
print("\n\n2. RETRACEMENT RATES BY DISTANCE BUCKET")
print("-" * 120)
print("Format: Rate% (Sample Size)")
print()

distance_data = {}
for bucket_min, bucket_max in DISTANCE_BUCKETS[:5]:  # Show first 5 buckets
    bucket_name = f"{bucket_min:.2f}-{bucket_max:.2f}%" if bucket_max != float('inf') else f"{bucket_min:.2f}%+"
    
    bucket_row = {}
    for level_name in level_time_order:
        if level_name in stats and bucket_name in stats[level_name]['overall']:
            data = stats[level_name]['overall'][bucket_name]
            bucket_row[level_name] = format_rate_sample(data['retraced_days'], data['total_days'])
        else:
            bucket_row[level_name] = "-"
    
    distance_data[bucket_name] = bucket_row

distance_df = pd.DataFrame(distance_data).T
print(distance_df.to_string())

# 3. RETRACEMENT RATES BY SESSION PATTERN - TRANSPOSED
print("\n\n3. RETRACEMENT RATES BY SESSION PATTERN")
print("-" * 120)
print("Format: Rate% (Sample Size)")
print()

patterns = ['london_engulfs_asia', 'asia_engulfs_london', 'london_partial_up', 'london_partial_down']
pattern_labels = {
    'london_engulfs_asia': 'London Engulfs Asia',
    'asia_engulfs_london': 'Asia Engulfs London',
    'london_partial_up': 'London Partial Up',
    'london_partial_down': 'London Partial Down'
}

pattern_data = {}
for pattern, label in pattern_labels.items():
    pattern_row = {}
    for level_name in level_time_order:
        if level_name in stats and pattern in stats[level_name]:
            pattern_stats = stats[level_name][pattern]
            total = sum(bucket_data['total_days'] for bucket_data in pattern_stats.values())
            retraced = sum(bucket_data['retraced_days'] for bucket_data in pattern_stats.values())
            pattern_row[level_name] = format_rate_sample(retraced, total)
        else:
            pattern_row[level_name] = "-"
    
    pattern_data[label] = pattern_row

pattern_df = pd.DataFrame(pattern_data).T
print(pattern_df.to_string())

# 4. TOP PATTERN × DISTANCE COMBINATIONS (across all levels)
print("\n\n4. TOP PATTERN × DISTANCE COMBINATIONS (20+ samples, sorted by rate)")
print("-" * 120)

all_combos = []
for level_name in stats.keys():
    for pattern in patterns:
        for bucket_min, bucket_max in DISTANCE_BUCKETS:
            bucket_name = f"{bucket_min:.2f}-{bucket_max:.2f}%" if bucket_max != float('inf') else f"{bucket_min:.2f}%+"
            combo_key = f"{pattern}_{bucket_name}"
            
            if combo_key in stats[level_name]:
                combo_stats = stats[level_name][combo_key]
                total = sum(bucket_data['total_days'] for bucket_data in combo_stats.values())
                retraced = sum(bucket_data['retraced_days'] for bucket_data in combo_stats.values())
                
                if total >= 20:
                    rate = (retraced / total) * 100
                    all_combos.append({
                        'Level': level_name,
                        'Pattern': pattern_labels[pattern],
                        'Distance': bucket_name,
                        'Total': total,
                        'Retraced': retraced,
                        'Rate_%': round(rate, 2)
                    })

if all_combos:
    combos_df = pd.DataFrame(all_combos)
    combos_df = combos_df.sort_values('Rate_%', ascending=False)
    print(combos_df.head(30).to_string(index=False))  # Top 30 combinations

# 5. TIMING ANALYSIS - WHEN DO RETRACEMENTS OCCUR?
print("\n\n5. TIMING ANALYSIS - When during 8am-4pm do retracements occur?")
print("-" * 120)
print("Showing top 5 levels by retracement rate (hourly buckets)")
print()

for i in range(min(5, len(level_summary_sorted))):
    level_name = level_summary_sorted[i]['Level']
    
    # Aggregate timing data
    overall_timing = defaultdict(int)
    total_retracements = 0
    
    for bucket_data in stats[level_name]['overall'].values():
        for time_slot, count in bucket_data['time_slots'].items():
            overall_timing[time_slot] += count
            total_retracements += count
    
    if total_retracements > 0:
        print(f"\n{level_name} (Total retracements: {total_retracements})")
        
        # Group into hourly buckets for concise display
        hourly_timing = defaultdict(int)
        for time_slot, count in overall_timing.items():
            hour = time_slot.split(':')[0]
            hourly_timing[hour] += count
        
        timing_rows = []
        for hour in sorted(hourly_timing.keys()):
            count = hourly_timing[hour]
            pct = (count / total_retracements) * 100
            timing_rows.append({
                'Hour': f"{hour}:00",
                'Count': count,
                'Pct_%': round(pct, 2)
            })
        
        timing_df = pd.DataFrame(timing_rows)
        print(timing_df.to_string(index=False))

# 6. ADVERSE EXCURSION ANALYSIS
print("\n\n6. ADVERSE EXCURSION ANALYSIS - How far does price move against the level before retracing?")
print("-" * 120)
print("For retracements that occurred, what was the maximum adverse move (in % terms)?")
print("This shows how much 'heat' you need to tolerate before the retracement happens.")
print()

# Analyze adverse excursion for top levels
for i in range(min(10, len(level_summary_sorted))):
    level_name = level_summary_sorted[i]['Level']
    
    # Aggregate adverse excursion data across all distance buckets
    adverse_excursion_totals = defaultdict(int)
    total_retracements = 0
    
    for bucket_data in stats[level_name]['overall'].values():
        for adverse_bucket, count in bucket_data['adverse_excursion_buckets'].items():
            adverse_excursion_totals[adverse_bucket] += count
            total_retracements += count
    
    if total_retracements > 0:
        print(f"\n{level_name} (Total retracements: {total_retracements})")
        
        adverse_rows = []
        for adverse_bucket in sorted(adverse_excursion_totals.keys()):
            count = adverse_excursion_totals[adverse_bucket]
            pct = (count / total_retracements) * 100
            adverse_rows.append({
                'Adverse_Move': adverse_bucket,
                'Count': count,
                'Pct_%': round(pct, 2)
            })
        
        if adverse_rows:
            adverse_df = pd.DataFrame(adverse_rows)
            print(adverse_df.to_string(index=False))

# 7. ADVERSE EXCURSION BY DISTANCE BUCKET
print("\n\n7. ADVERSE EXCURSION BY DISTANCE FROM LEVEL")
print("-" * 120)
print("How does adverse excursion vary based on how far away the level is?")
print("Showing data for top 3 levels")
print()

for i in range(min(3, len(level_summary_sorted))):
    level_name = level_summary_sorted[i]['Level']
    
    print(f"\n{level_name}:")
    print("-" * 120)
    
    # For each distance bucket, show adverse excursion distribution
    for bucket_min, bucket_max in DISTANCE_BUCKETS[:4]:  # First 4 distance buckets
        bucket_name = f"{bucket_min:.2f}-{bucket_max:.2f}%" if bucket_max != float('inf') else f"{bucket_min:.2f}%+"
        
        if bucket_name in stats[level_name]['overall']:
            bucket_data = stats[level_name]['overall'][bucket_name]
            
            if bucket_data['retraced_days'] > 0:
                print(f"\nDistance from level: {bucket_name} (Retracements: {bucket_data['retraced_days']})")
                
                adverse_rows = []
                for adverse_bucket, count in bucket_data['adverse_excursion_buckets'].items():
                    pct = (count / bucket_data['retraced_days']) * 100
                    adverse_rows.append({
                        'Adverse_Move': adverse_bucket,
                        'Count': count,
                        'Pct_%': round(pct, 2)
                    })
                
                if adverse_rows:
                    adverse_df = pd.DataFrame(adverse_rows)
                    print(adverse_df.to_string(index=False))

print("\n\n" + "=" * 120)
print("ANALYSIS COMPLETE")
print("=" * 120)
print(f"\nFull dataset exported to: {output_path}")

# ========================================================================
# GENERATE FORMATTED TABLES AS IMAGES
# ========================================================================

print("\n\nGenerating formatted table visualizations...")

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle

# Set up matplotlib for clean monochrome tables
plt.rcParams['font.family'] = 'monospace'
plt.rcParams['font.size'] = 8

output_dir = r"C:\Users\tammm\NQ and ES analysis"

# Helper function to create clean tables
def create_table_image(df, title, filename, figsize=(16, 10), highlight_top=None):
    """Create a clean monochrome table and save as PNG"""
    fig, ax = plt.subplots(figsize=figsize)
    ax.axis('tight')
    ax.axis('off')
    
    # Add title
    fig.suptitle(title, fontsize=14, fontweight='bold', y=0.98)
    
    # Create table
    table = ax.table(cellText=df.values,
                     colLabels=df.columns,
                     cellLoc='center',
                     loc='center',
                     bbox=[0, 0, 1, 1])
    
    # Style the table
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 1.5)
    
    # Header styling - dark gray background, white text
    for (i, j), cell in table.get_celld().items():
        if i == 0:  # Header row
            cell.set_facecolor('#2C3E50')
            cell.set_text_props(weight='bold', color='white')
        else:
            # Alternate row colors for readability
            if i % 2 == 0:
                cell.set_facecolor('#F8F9FA')
            else:
                cell.set_facecolor('#FFFFFF')
            
            # Highlight top performers if specified
            if highlight_top and i <= highlight_top:
                cell.set_facecolor('#E8F4F8')
        
        cell.set_edgecolor('#BDC3C7')
        cell.set_linewidth(0.5)
    
    plt.savefig(f"{output_dir}/{filename}", dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"  Saved: {filename}")

# 1. OVERALL SUMMARY TABLE
print("\n1. Creating Overall Summary table...")
summary_data = []
for level_name in level_time_order:
    if level_name in stats:
        overall_stats = stats[level_name]['overall']
        total = sum(bucket_data['total_days'] for bucket_data in overall_stats.values())
        retraced = sum(bucket_data['retraced_days'] for bucket_data in overall_stats.values())
        rate = (retraced / total) * 100 if total > 0 else 0
        
        summary_data.append({
            'Level': level_name,
            'Total Days': total,
            'Retraced': retraced,
            'Rate %': f"{rate:.2f}%"
        })

summary_df = pd.DataFrame(summary_data)
create_table_image(summary_df, 
                   'Overall Retracement Rates by Level (8am-4pm NY Futures Session)',
                   'table_1_overall_summary.png',
                   figsize=(12, 14),
                   highlight_top=5)

# 2. DISTANCE BUCKET ANALYSIS - Transposed (top 10 levels only for readability)
print("\n2. Creating Distance Bucket Analysis tables...")

# Top 10 levels
top_10_levels = [item['Level'] for item in level_summary_sorted[:10]]

for bucket_min, bucket_max in DISTANCE_BUCKETS[:5]:
    bucket_name = f"{bucket_min:.2f}-{bucket_max:.2f}%" if bucket_max != float('inf') else f"{bucket_min:.2f}%+"
    
    bucket_data = []
    for level_name in top_10_levels:
        if level_name in stats and bucket_name in stats[level_name]['overall']:
            data = stats[level_name]['overall'][bucket_name]
            rate = (data['retraced_days'] / data['total_days']) * 100 if data['total_days'] > 0 else 0
            bucket_data.append({
                'Level': level_name,
                'Total Days': data['total_days'],
                'Retraced': data['retraced_days'],
                'Rate %': f"{rate:.1f}%"
            })
    
    if bucket_data:
        bucket_df = pd.DataFrame(bucket_data)
        safe_name = bucket_name.replace('%', 'pct').replace('-', '_').replace('+', 'plus')
        create_table_image(bucket_df,
                          f'Retracement Rates - Distance Bucket: {bucket_name}',
                          f'table_2_distance_{safe_name}.png',
                          figsize=(10, 6))

# 3. SESSION PATTERN ANALYSIS - Create separate table for each pattern
print("\n3. Creating Session Pattern Analysis tables...")

patterns = ['london_engulfs_asia', 'asia_engulfs_london', 'london_partial_up', 'london_partial_down']
pattern_labels = {
    'london_engulfs_asia': 'London Engulfs Asia',
    'asia_engulfs_london': 'Asia Engulfs London',
    'london_partial_up': 'London Partial Engulf Up',
    'london_partial_down': 'London Partial Engulf Down'
}

for pattern, label in pattern_labels.items():
    pattern_data = []
    
    for level_name in level_time_order:
        if level_name in stats and pattern in stats[level_name]:
            pattern_stats = stats[level_name][pattern]
            total = sum(bucket_data['total_days'] for bucket_data in pattern_stats.values())
            retraced = sum(bucket_data['retraced_days'] for bucket_data in pattern_stats.values())
            
            if total > 0:
                rate = (retraced / total) * 100
                pattern_data.append({
                    'Level': level_name,
                    'Total Days': total,
                    'Retraced': retraced,
                    'Rate %': f"{rate:.1f}%"
                })
    
    if pattern_data:
        pattern_df = pd.DataFrame(pattern_data)
        safe_pattern = pattern.replace('_', '')
        create_table_image(pattern_df,
                          f'Retracement Rates by Pattern: {label}',
                          f'table_3_pattern_{safe_pattern}.png',
                          figsize=(12, 14))

# 4. TOP PATTERN × DISTANCE COMBINATIONS
print("\n4. Creating Pattern × Distance Combinations table...")

all_combos = []
for level_name in stats.keys():
    for pattern in patterns:
        for bucket_min, bucket_max in DISTANCE_BUCKETS:
            bucket_name = f"{bucket_min:.2f}-{bucket_max:.2f}%" if bucket_max != float('inf') else f"{bucket_min:.2f}%+"
            combo_key = f"{pattern}_{bucket_name}"
            
            if combo_key in stats[level_name]:
                combo_stats = stats[level_name][combo_key]
                total = sum(bucket_data['total_days'] for bucket_data in combo_stats.values())
                retraced = sum(bucket_data['retraced_days'] for bucket_data in combo_stats.values())
                
                if total >= 20:
                    rate = (retraced / total) * 100
                    all_combos.append({
                        'Level': level_name,
                        'Pattern': pattern_labels[pattern],
                        'Distance': bucket_name,
                        'Total': total,
                        'Retraced': retraced,
                        'Rate %': f"{rate:.1f}%"
                    })

if all_combos:
    combos_df = pd.DataFrame(all_combos)
    combos_df = combos_df.sort_values('Rate %', ascending=False, key=lambda x: x.str.rstrip('%').astype(float))
    
    # Top 30
    create_table_image(combos_df.head(30),
                      'Top 30 Pattern × Distance Combinations (20+ samples)',
                      'table_4_top_combinations.png',
                      figsize=(14, 12),
                      highlight_top=10)

# 5. TIMING ANALYSIS - When do retracements occur?
print("\n5. Creating Timing Analysis tables...")

for i in range(min(5, len(level_summary_sorted))):
    level_name = level_summary_sorted[i]['Level']
    
    # Aggregate timing data
    overall_timing = defaultdict(int)
    total_retracements = 0
    
    for bucket_data in stats[level_name]['overall'].values():
        for time_slot, count in bucket_data['time_slots'].items():
            overall_timing[time_slot] += count
            total_retracements += count
    
    if total_retracements > 0:
        # Group into hourly buckets
        hourly_timing = defaultdict(int)
        for time_slot, count in overall_timing.items():
            hour = time_slot.split(':')[0]
            hourly_timing[hour] += count
        
        timing_data = []
        for hour in sorted(hourly_timing.keys()):
            count = hourly_timing[hour]
            pct = (count / total_retracements) * 100
            timing_data.append({
                'Hour': f"{hour}:00",
                'Count': count,
                'Percentage': f"{pct:.2f}%"
            })
        
        timing_df = pd.DataFrame(timing_data)
        safe_level = level_name.replace(':', '').replace('_', '')
        create_table_image(timing_df,
                          f'Timing Analysis: {level_name} ({total_retracements} retracements)',
                          f'table_5_timing_{safe_level}.png',
                          figsize=(8, 5))

# 6. ADVERSE EXCURSION ANALYSIS
print("\n6. Creating Adverse Excursion Analysis tables...")

for i in range(min(10, len(level_summary_sorted))):
    level_name = level_summary_sorted[i]['Level']
    
    # Aggregate adverse excursion data
    adverse_excursion_totals = defaultdict(int)
    total_retracements = 0
    
    for bucket_data in stats[level_name]['overall'].values():
        for adverse_bucket, count in bucket_data['adverse_excursion_buckets'].items():
            adverse_excursion_totals[adverse_bucket] += count
            total_retracements += count
    
    if total_retracements > 0:
        adverse_data = []
        for adverse_bucket in sorted(adverse_excursion_totals.keys()):
            count = adverse_excursion_totals[adverse_bucket]
            pct = (count / total_retracements) * 100
            adverse_data.append({
                'Adverse Move': adverse_bucket,
                'Count': count,
                'Percentage': f"{pct:.2f}%"
            })
        
        if adverse_data:
            adverse_df = pd.DataFrame(adverse_data)
            safe_level = level_name.replace(':', '').replace('_', '')
            create_table_image(adverse_df,
                              f'Adverse Excursion: {level_name} ({total_retracements} retracements)',
                              f'table_6_adverse_{safe_level}.png',
                              figsize=(8, 6))

# 7. ADVERSE EXCURSION BY DISTANCE (Top 3 levels only)
print("\n7. Creating Adverse Excursion by Distance tables...")

for i in range(min(3, len(level_summary_sorted))):
    level_name = level_summary_sorted[i]['Level']
    safe_level = level_name.replace(':', '').replace('_', '')
    
    for bucket_min, bucket_max in DISTANCE_BUCKETS[:4]:
        bucket_name = f"{bucket_min:.2f}-{bucket_max:.2f}%" if bucket_max != float('inf') else f"{bucket_min:.2f}%+"
        
        if bucket_name in stats[level_name]['overall']:
            bucket_data = stats[level_name]['overall'][bucket_name]
            
            if bucket_data['retraced_days'] > 0:
                adverse_data = []
                for adverse_bucket, count in bucket_data['adverse_excursion_buckets'].items():
                    pct = (count / bucket_data['retraced_days']) * 100
                    adverse_data.append({
                        'Adverse Move': adverse_bucket,
                        'Count': count,
                        'Percentage': f"{pct:.2f}%"
                    })
                
                if adverse_data:
                    adverse_df = pd.DataFrame(adverse_data)
                    safe_bucket = bucket_name.replace('%', 'pct').replace('-', '_').replace('+', 'plus')
                    create_table_image(adverse_df,
                                      f'{level_name} - Distance {bucket_name} ({bucket_data["retraced_days"]} retracements)',
                                      f'table_7_adverse_dist_{safe_level}_{safe_bucket}.png',
                                      figsize=(8, 6))

print("\n" + "=" * 120)
print("TABLE GENERATION COMPLETE")
print("=" * 120)
print(f"\nAll tables saved to: {output_dir}")
print("Check for files starting with 'table_'")

Processing hourly level retracement data...
Tracking 28 levels per day
Distance buckets: 9

Processed 500 days...
Processed 1000 days...
Processed 1500 days...
Processed 2000 days...
Processed 2500 days...
Processed 3000 days...
Processed 3500 days...

Processed 3785 days total

ANALYSIS COMPLETE - Generating Reports

Detailed data exported to: C:\Users\tammm\NQ and ES analysis\hourly_level_retracements.csv



TOP PERFORMING LEVELS - OVERALL

 Rank            Level  Total_Days  Retraced  Rate_%
    1 07:00am_midpoint        3130      2948   94.19
    2     07:00am_open        3127      2766   88.46
    3 06:00am_midpoint        3130      2731   87.25
    4     06:00am_open        3125      2638   84.42
    5 05:00am_midpoint        3130      2640   84.35
    6     05:00am_open        3124      2584   82.71
    7 04:00am_midpoint        3130      2558   81.73
    8     04:00am_open        3129      2490   79.58
    9 03:00am_midpoint        3130      2468   78.85
   10     03:00am_open 


1. Creating Overall Summary table...
  Saved: table_1_overall_summary.png

2. Creating Distance Bucket Analysis tables...
  Saved: table_2_distance_0.00_0.25pct.png
  Saved: table_2_distance_0.25_0.50pct.png
  Saved: table_2_distance_0.50_0.75pct.png
  Saved: table_2_distance_0.75_1.00pct.png
  Saved: table_2_distance_1.00_1.50pct.png

3. Creating Session Pattern Analysis tables...
  Saved: table_3_pattern_londonengulfsasia.png
  Saved: table_3_pattern_asiaengulfslondon.png
  Saved: table_3_pattern_londonpartialup.png
  Saved: table_3_pattern_londonpartialdown.png

4. Creating Pattern × Distance Combinations table...
  Saved: table_4_top_combinations.png

5. Creating Timing Analysis tables...
  Saved: table_5_timing_0700ammidpoint.png
  Saved: table_5_timing_0700amopen.png
  Saved: table_5_timing_0600ammidpoint.png
  Saved: table_5_timing_0600amopen.png
  Saved: table_5_timing_0500ammidpoint.png

6. Creating Adverse Excursion Analysis tables...
  Saved: table_6_adverse_0700ammidpoint.

In [7]:
"""
Hourly Candle Retracement Analysis — NQ Futures (All 24h incl. Globex)
=======================================================================
For every completed hourly candle, checks whether the NEXT candle (or
subsequent candles within a look-forward window) break the prior hour's
high or low, then retraces back to the prior hour's midpoint or open.

Break definitions tracked separately:
  - Intrabar touch  : any bar's high > prior_high  (or low < prior_low)
  - Close break     : bar closes above prior_high   (or below prior_low)

Retracement definitions tracked separately:
  - Touch retracement : a subsequent bar's low <= target  (high >= target for low breaks)
  - Close retracement : a subsequent bar closes at/through the target

Outputs a full summary table per hour-of-day (0–23 ET) and an overall
summary, plus a CSV of every individual event for further slicing.
"""

import pandas as pd
import numpy as np
from pathlib import Path

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
FILE_PATH      = r"C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv"
LOOKFORWARD_H  = 4          # how many hourly candles forward to look for retracement
OUTPUT_EVENTS  = r"C:\Users\tammm\NQ and ES analysis\hourly_retracement_events.csv"
OUTPUT_SUMMARY = r"C:\Users\tammm\NQ and ES analysis\hourly_retracement_summary.csv"

# ---------------------------------------------------------------------------
# 1. Load 1-min data  (replicates existing project pattern exactly)
# ---------------------------------------------------------------------------
print("Loading data...")
df = pd.read_csv(FILE_PATH)
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df.set_index('Datetime', inplace=True)

start_date = pd.to_datetime('2013-01-01')
end_date   = pd.to_datetime('2025-12-31')
df = df.loc[start_date:end_date]

# NY trading day grouping: 18:00–17:00 ET  (offset by 7h so midnight = 19:00 prev day)
df['NY_Trading_Day'] = (df.index - pd.Timedelta(hours=7)).normalize()

print(f"  Loaded {len(df):,} 1-min bars  |  {df['NY_Trading_Day'].nunique():,} trading days")

# ---------------------------------------------------------------------------
# 2. Resample to hourly OHLC  (keep ET timezone labelling)
# ---------------------------------------------------------------------------
print("Resampling to hourly...")
hourly = df[['Open', 'High', 'Low', 'Close']].resample('1h').agg({
    'Open':  'first',
    'High':  'max',
    'Low':   'min',
    'Close': 'last'
}).dropna()

hourly['Mid']        = (hourly['High'] + hourly['Low']) / 2
hourly['Hour_ET']    = hourly.index.hour          # 0–23, timestamps in ET
hourly['NY_Trading_Day'] = (hourly.index - pd.Timedelta(hours=7)).normalize()

print(f"  {len(hourly):,} hourly bars across {hourly['NY_Trading_Day'].nunique():,} sessions")

# ---------------------------------------------------------------------------
# 3. Walk every hourly bar and detect break + retracement
# ---------------------------------------------------------------------------
print("Analysing breaks and retracements...")

events = []

bars = hourly.reset_index()   # flat list, integer-indexed for easy look-ahead

for i in range(len(bars) - 1):
    prior = bars.iloc[i]

    prior_high = prior['High']
    prior_low  = prior['Low']
    prior_mid  = prior['Mid']
    prior_open = prior['Open']
    prior_time = prior['Datetime']
    hour_et    = prior['Hour_ET']
    ny_day     = prior['NY_Trading_Day']

    # Look-forward window: up to LOOKFORWARD_H candles ahead
    end_idx = min(i + 1 + LOOKFORWARD_H, len(bars))
    forward = bars.iloc[i+1 : end_idx]

    if forward.empty:
        continue

    # ---- Detect break type for HIGH ----------------------------------------
    for direction in ('HIGH', 'LOW'):
        break_level  = prior_high if direction == 'HIGH' else prior_low
        target_mid   = prior_mid
        target_open  = prior_open

        # Which forward bars qualify as a break?
        if direction == 'HIGH':
            touch_break_mask = forward['High']  > break_level
            close_break_mask = forward['Close'] > break_level
        else:
            touch_break_mask = forward['Low']   < break_level
            close_break_mask = forward['Close'] < break_level

        for break_type, break_mask in (('touch', touch_break_mask),
                                       ('close', close_break_mask)):

            if not break_mask.any():
                # No break of this type occurred → still record as no-break
                events.append({
                    'prior_datetime'    : prior_time,
                    'hour_et'           : hour_et,
                    'ny_trading_day'    : ny_day,
                    'direction'         : direction,
                    'break_type'        : break_type,
                    'broke'             : False,
                    'break_bar_offset'  : np.nan,
                    'mid_touch_retrace' : np.nan,
                    'mid_close_retrace' : np.nan,
                    'open_touch_retrace': np.nan,
                    'open_close_retrace': np.nan,
                    'prior_high'        : prior_high,
                    'prior_low'         : prior_low,
                    'prior_mid'         : prior_mid,
                    'prior_open'        : prior_open,
                })
                continue

            # First bar that constitutes a break
            first_break_idx  = break_mask.idxmax()
            break_bar_offset = first_break_idx - (i + 1) + 1   # 1 = next bar

            # Post-break look-forward (bars after the break bar, still within window)
            post_break = forward.loc[first_break_idx:]   # inclusive of break bar

            # Retracement checks
            # For a HIGH break, retrace means price comes back DOWN to target
            # For a LOW  break, retrace means price comes back UP   to target
            def retrace_touch(df_fwd, target, direction):
                if direction == 'HIGH':
                    return (df_fwd['Low'] <= target).any()
                else:
                    return (df_fwd['High'] >= target).any()

            def retrace_close(df_fwd, target, direction):
                if direction == 'HIGH':
                    return (df_fwd['Close'] <= target).any()
                else:
                    return (df_fwd['Close'] >= target).any()

            events.append({
                'prior_datetime'    : prior_time,
                'hour_et'           : hour_et,
                'ny_trading_day'    : ny_day,
                'direction'         : direction,
                'break_type'        : break_type,
                'broke'             : True,
                'break_bar_offset'  : break_bar_offset,
                'mid_touch_retrace' : retrace_touch(post_break, target_mid,  direction),
                'mid_close_retrace' : retrace_close(post_break, target_mid,  direction),
                'open_touch_retrace': retrace_touch(post_break, target_open, direction),
                'open_close_retrace': retrace_close(post_break, target_open, direction),
                'prior_high'        : prior_high,
                'prior_low'         : prior_low,
                'prior_mid'         : prior_mid,
                'prior_open'        : prior_open,
            })

events_df = pd.DataFrame(events)
print(f"  {len(events_df):,} event records generated")

# ---------------------------------------------------------------------------
# 4. Summary statistics
# ---------------------------------------------------------------------------
print("Building summary tables...")

broke_df = events_df[events_df['broke'] == True].copy()

# --- Overall summary --------------------------------------------------------
def summarise(group):
    rows = []
    for (direction, break_type), sub in group:
        n = len(sub)
        if n == 0:
            continue
        rows.append({
            'direction'              : direction,
            'break_type'            : break_type,
            'n_breaks'              : n,
            'mid_touch_retrace_%'   : round(sub['mid_touch_retrace'].mean()  * 100, 1),
            'mid_close_retrace_%'   : round(sub['mid_close_retrace'].mean()  * 100, 1),
            'open_touch_retrace_%'  : round(sub['open_touch_retrace'].mean() * 100, 1),
            'open_close_retrace_%'  : round(sub['open_close_retrace'].mean() * 100, 1),
            'median_break_bar'      : round(sub['break_bar_offset'].median(), 1),
        })
    return pd.DataFrame(rows)

overall = summarise(broke_df.groupby(['direction', 'break_type']))

# --- By hour-of-day ---------------------------------------------------------
hour_rows = []
for (hour, direction, break_type), sub in broke_df.groupby(['hour_et', 'direction', 'break_type']):
    n = len(sub)
    if n < 5:   # skip tiny samples
        continue

    # Also compute break rate vs total bars at that hour
    total_at_hour = len(events_df[
        (events_df['hour_et']    == hour) &
        (events_df['direction']  == direction) &
        (events_df['break_type'] == break_type)
    ])

    hour_rows.append({
        'hour_et'               : hour,
        'direction'             : direction,
        'break_type'            : break_type,
        'total_bars'            : total_at_hour,
        'n_breaks'              : n,
        'break_rate_%'          : round(n / total_at_hour * 100, 1) if total_at_hour else np.nan,
        'mid_touch_retrace_%'   : round(sub['mid_touch_retrace'].mean()  * 100, 1),
        'mid_close_retrace_%'   : round(sub['mid_close_retrace'].mean()  * 100, 1),
        'open_touch_retrace_%'  : round(sub['open_touch_retrace'].mean() * 100, 1),
        'open_close_retrace_%'  : round(sub['open_close_retrace'].mean() * 100, 1),
        'median_break_bar'      : round(sub['break_bar_offset'].median(), 1),
    })

hourly_summary = pd.DataFrame(hour_rows).sort_values(['hour_et', 'direction', 'break_type'])

# ---------------------------------------------------------------------------
# 5. Print results to console
# ---------------------------------------------------------------------------
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', '{:.1f}'.format)

print("\n" + "="*80)
print("OVERALL RETRACEMENT RATES  (all hours combined)")
print("="*80)
print(overall.to_string(index=False))

print("\n" + "="*80)
print(f"BY HOUR OF DAY (ET)  —  look-forward window: {LOOKFORWARD_H}h")
print("="*80)
print()

# Pivot-style print: for each hour show all 4 combos side by side
for hour in sorted(hourly_summary['hour_et'].unique()):
    h_str = f"{hour:02d}:00 ET"
    sub = hourly_summary[hourly_summary['hour_et'] == hour]
    print(f"  {h_str}")
    print(f"  {'Direction':<8} {'Break':>10} {'Bars':>6} {'Breaks':>7} {'Brk%':>6} "
          f"{'Mid-Touch':>10} {'Mid-Close':>10} {'Opn-Touch':>10} {'Opn-Close':>10} {'Med Bar':>8}")
    for _, r in sub.iterrows():
        print(f"  {r['direction']:<8} {r['break_type']:>10} {int(r['total_bars']):>6} "
              f"{int(r['n_breaks']):>7} {r['break_rate_%']:>6.1f}% "
              f"{r['mid_touch_retrace_%']:>9.1f}% {r['mid_close_retrace_%']:>9.1f}% "
              f"{r['open_touch_retrace_%']:>9.1f}% {r['open_close_retrace_%']:>9.1f}% "
              f"{r['median_break_bar']:>8.1f}")
    print()

# ---------------------------------------------------------------------------
# 6. Save to CSV
# ---------------------------------------------------------------------------
events_df.to_csv(OUTPUT_EVENTS, index=False)
hourly_summary.to_csv(OUTPUT_SUMMARY, index=False)

print(f"\nSaved events  → {OUTPUT_EVENTS}")
print(f"Saved summary → {OUTPUT_SUMMARY}")
print("\nDone.")

Loading data...
  Loaded 4,161,727 1-min bars  |  3,786 trading days
Resampling to hourly...
  72,371 hourly bars across 3,786 sessions
Analysing breaks and retracements...
  289,480 event records generated
Building summary tables...

OVERALL RETRACEMENT RATES  (all hours combined)
direction break_type  n_breaks  mid_touch_retrace_%  mid_close_retrace_%  open_touch_retrace_%  open_close_retrace_%  median_break_bar
     HIGH      close     38719                  0.0                  0.0                   0.0                   0.0               2.0
     HIGH      touch     51954                  0.0                  0.0                   0.0                   0.0               1.0
      LOW      close     32781                  0.0                  0.0                   0.0                   0.0               2.0
      LOW      touch     47676                  0.0                  0.0                   0.0                   0.0               1.0

BY HOUR OF DAY (ET)  —  look-forward wind


Saved events  → C:\Users\tammm\NQ and ES analysis\hourly_retracement_events.csv
Saved summary → C:\Users\tammm\NQ and ES analysis\hourly_retracement_summary.csv

Done.
